#### 检查工作路径

In [ ]:
import os
print(os.getcwd())

#### 检查all_peptide_data行数是否正确，并获得id和list中编号的对应字典

In [ ]:
import json

json_path = './Data/all_peptides_data.json'

# 打开并读取JSON文件
with open(json_path, 'r', encoding='utf-8') as file:
    data = json.load(file)  # 将JSON内容加载为Python列表

# 打印读取的数据
print("number of AMPs: ", len(data))
id_dict={}
for i, AMP in enumerate(data):
    id_dict[AMP['id']] = i
    
print("id_dict:", id_dict)

#### 看第一个peptide

In [ ]:
print(json.dumps(data[id_dict[11510]], indent=4, ensure_ascii=False))

#### 看第一个peptide的特定性质:synergies

In [ ]:
print('id:', data[0]['id'])
print('smiles:\n', json.dumps(data[33]['smiles'], indent=4, ensure_ascii=False))
print('synergies:\n', json.dumps(data[33]['synergies'], indent=4, ensure_ascii=False))

In [ ]:
target_AMP = 33
print(f'id:{data[target_AMP]['id']},\ttype:{data[target_AMP]['complexity']['name']},\tsequence:{data[target_AMP]['sequence']},\tmonomers:{data[target_AMP]['monomers']}\n')

target_AMP = 0
for i in range(len(data[target_AMP]['monomers'])):
    print(f'id:{data[target_AMP]['id']},\ttype:{data[target_AMP]['complexity']['name']},\tsequence:{data[target_AMP]['sequence']},\tmonomers {i}:{data[target_AMP]['monomers'][i]['sequence']}')

#### 检查有没有unknown氨基酸前续步骤，检查'sequence'部分没有值或者是list长度为0的AMP是否都是multimer和multi-peptide

In [ ]:
from tqdm import tqdm
multimer = 0
multimer_no_seq = 0
multi_peptide = 0
for AMP in tqdm(data):
    if AMP['complexity']['name'] == 'Multimer':
        multimer += 1
        if (AMP['sequence'] is not None) and len(AMP['sequence']) > 0:
            for i in range(len(AMP['monomers'])):
                print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence']},\tmonomer{i}:{AMP['monomers'][i]['sequence']}')
            print('\n')
    if AMP['sequence'] is None :
        # print('None', AMP['complexity']['name'])
        if AMP['complexity']['name'] == 'Multimer':
            multimer_no_seq += 1
        elif AMP['complexity']['name'] == 'Multi-Peptide':
            multi_peptide += 1
        else:
            print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence: None,\tmonomer{0}:{AMP['monomers'][0]['sequence']}')
    elif len(AMP['sequence']) == 0:
        # print('Zero', AMP['complexity']['name'])
        if AMP['complexity']['name'] == 'Multimer':
            multimer_no_seq += 1
        elif AMP['complexity']['name'] == 'Multi-Peptide':
            multi_peptide += 1
        else:
            print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence']},\tmonomer{0}:{AMP['monomers'][0]['sequence']}')
print(f'multimer: {multimer}')
print(f'multimer_no_seq: {multimer_no_seq}')
print(f'multi_peptide: {multi_peptide}')

上述总结，所有的Multimer和Multi-Peptides都有至少2个monomer

#### 检查有哪些字符出现在sequence里面过

In [ ]:
unique_characters = set()
unique_uppercase_characters = set()
unique_lowercase_characters = set()
for AMP in data:
    if AMP['complexity']['name'] == 'Monomer':
        for char in AMP['sequence']:
            unique_characters.add(char)
    else:
        for monomer in AMP['monomers']:
            for char in monomer['sequence']:
                unique_characters.add(char)
print("所有出现过的字符:", unique_characters)
for char in unique_characters:
    if char.isupper():
        unique_uppercase_characters.add(char)
    if char.islower():
        unique_lowercase_characters.add(char)
sorted_uppercase_characters = sorted(unique_uppercase_characters)
sorted_lowercase_characters = sorted(unique_lowercase_characters)
print(f"uppercase: {sorted_uppercase_characters}\tlength: {len(sorted_uppercase_characters)}")
print("lowercase:", sorted_lowercase_characters)

看起来O（Pyrrolysine）没有手性分子出现在这里，大写是L-小写是D-

#### 检查 [可选字符] 出现在了哪些序列里，出现了多少次

In [ ]:
import re
count_X_x = 0
check_char = 'O'
print(f'\n###############\nchecked character: "{check_char}"')
for AMP in data:
    if AMP['complexity']['name'] == 'Monomer':
        if re.search(r"[O]", AMP['sequence']):
            print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence']},\tmonomer:{AMP['monomers']}')
            count_X_x += 1
    else:
        # print(1)
        for i, monomer in enumerate(AMP['monomers']):
            if re.search(r"[O]", monomer['sequence']):
                print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence'] if AMP['sequence'] is not None else None},\tmonomer {i} sequence:{monomer['sequence']}')
                count_X_x += 1
print(f'O appeared in {count_X_x} sequences')

空格' '出现在开头或者结尾，去掉就好  
带有O的是Ornithine（鸟氨酸），一种非标准氨基酸

#### 单独检查X和x出现了多少次

In [ ]:
count_X_x = 0
check_char = '[Xx]'
print(f'\n###############\nchecked character: "{check_char}"')
for AMP in data:
    if AMP['complexity']['name'] == 'Monomer':
        if re.search(r"[Xx]", AMP['sequence']):
            print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence']},\tmonomer:{AMP['monomers']}')
            count_X_x += 1
    else:
        # print(1)
        for i, monomer in enumerate(AMP['monomers']):
            if re.search(r"[Xx]", monomer['sequence']):
                print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence'] if AMP['sequence'] is not None else None},\tmonomer {i} sequence:{monomer['sequence']}')
                count_X_x += 1
print(f'X and x appeared in {count_X_x} sequences')

#### 检查有多少AMP有intrachain bonds

In [ ]:
cycle_count = 0
for AMP in data:
    if len(AMP.get('intrachainBonds', [])) > 0:
        cycle_count += 1
print(f'cycle count: {cycle_count}')

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
# import selfies as sf
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import MolDrawOptions

# 使用从 OPSIN 获得的 SMILES
smiles = "[C@@H](C(C)C)NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](CCCCN)NC(=O)[C@@H](C)NC(=O)[C@@H](C)NC(=O)[C@@H](Cc1ccccc1)NC(=O)[C@@H](Cc1c[nH]cn1)NC(=O)[C@@H](CCSC)NC(=O)[C@@H](CCCNC(=N)N)NC(=O)[C@@H](CCCCN)NC(=O)[C@@H](CC(C)C)NC(=O)[C@@H](Cc1ccccc1)NC(=O)[C@@H](CCCNC(=N)N)NC(=O)CNC(=O)[C@@H](Cc1c[nH]cn1)NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(=O)[C@@H](C(C)C)NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(=O)[C@@H](Cc1c[nH]c2ccccc12)"
# smiles = "[C][=O][N]"
# C[N+](C)(C)CCCC[C@@H](C(=O)[O-])[NH3+]
# smiles = sf.decoder(smiles)
# print(sf.encoder(smiles))
# 创建分子对象
mol = Chem.MolFromSmiles(smiles)

rdDepictor.SetPreferCoordGen(True)
rdDepictor.Compute2DCoords(mol)

opts = MolDrawOptions()
opts.reduceOverlap = True

# 绘制分子结构
img = Draw.MolToImage(mol, size=(2000, 1000))
display(img)

#### 绘制SMILES对应的分子图

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
import selfies as sf
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import MolDrawOptions

# 使用从 OPSIN 获得的 SMILES
smiles = "Nc1nc(SCc2csc(n2)-c2ccc(Cl)cc2)c(C#N)c(-c2ccc(OCCO)cc2)c1C#N"
# smiles = "[C][=O][N]"
# C[N+](C)(C)CCCC[C@@H](C(=O)[O-])[NH3+]
# smiles = sf.decoder(smiles)
print(smiles)
# 创建分子对象
mol = Chem.MolFromSmiles(smiles)

rdDepictor.SetPreferCoordGen(True)
rdDepictor.Compute2DCoords(mol)

opts = MolDrawOptions()
opts.reduceOverlap = True

# 绘制分子结构
img = Draw.MolToImage(mol, size=(2000, 1000))
display(img)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
import selfies as sf
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import MolDrawOptions

# 使用从 OPSIN 获得的 SMILES
smiles = "[N][C@@H1][Branch1][=Branch1][C][C][C][C][N][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][C][C][Ring1][Branch1][C][=Branch1][C][=O][N][C@@H1][Branch1][#Branch1][C@H1][Branch1][Ring1][C][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][=C][C][C][C][C][N][O][C][C][O][C][C][O][C][C][=Branch1][C][=O][N][C@@H1][Branch1][=N][C][C][=C][C][=C][Branch1][C][O][C][=C][Ring1][#Branch1][C][=Branch1][C][=O][N][C@@H1][Branch1][Branch1][C][C][C][=O][C][=Branch1][C][=O][N][C@@H1][Branch1][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][=N][C][C][=C][C][=C][Branch1][C][O][C][=C][Ring1][#Branch1][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][C][C][C][N][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch2][C][C][=C][N][C][=N][Ring1][Branch1][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][Branch1][C][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][C][C][Ring1][Branch1][C][=Branch1][C][=O][N][C][C][=Branch1][C][=O][N][C@@H1][Branch1][Ring1][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][#Branch2][C][C][C][N][C][=Branch1][C][=N][N][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][C][C][C][N][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C@H1][Branch1][C][O][C][C][=Branch1][C][=O][N][C@@H1][Branch1][#Branch1][C@H1][Branch1][Ring1][C][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][P][C][C][=Branch1][Ring1][=C][N][C][=C][Ring1][Ring1][C][=C][C][=C][Ring1][=Branch1][C][=Branch1][C][=O][N][C@@H1][Branch1][Branch1][C][C][S][C][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][C][C][Ring1][Branch1][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][C][C][C][N][C][=Branch1][C][=O][N][C@@H1][Branch1][=N][C][C][=C][C][=C][Branch1][Branch1][C][=C][Ring1][=Branch1][N][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][Branch1][C][C][C][C][=Branch1][C][=O][N][C@@H1][Branch1][=Branch1][C][C][C][C][N][C][=Branch1][C][=O][N]"
# smiles = "[C][=O][N]"
# C[N+](C)(C)CCCC[C@@H](C(=O)[O-])[NH3+]
smiles = sf.decoder(smiles)
print(smiles)
# 创建分子对象
mol = Chem.MolFromSmiles(smiles)

rdDepictor.SetPreferCoordGen(True)
rdDepictor.Compute2DCoords(mol)

opts = MolDrawOptions()
opts.reduceOverlap = True

# 绘制分子结构
img = Draw.MolToImage(mol, size=(2000, 1000))
display(img)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw


# 使用从 OPSIN 获得的 SMILES
smiles = "CCC1NC(=O)[C@H](C(C)C)NC(=O)CNC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CCCN=C(N)N)NC(=O)C(CC)NC(=O)[C@H](C(C)C)NC(=O)[C@@H]2CSSC[C@H](NC(=O)[C@H](CCCN=C(N)N)NC1=O)C(=O)N[C@@H](C(C)C)C(=O)NC(CC)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCN=C(N)N)C(=O)NCC(=O)N[C@@H](C(C)C)C(=O)NC(CC)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N2"  # Allo-Thr: CC(C(C(=O)[O-])[NH3+])O

# 创建分子对象
mol = Chem.MolFromSmiles(smiles)
drawer = Draw.MolDraw2DSVG(1500, 1000)
drawer.DrawMolecule(mol)
drawer.FinishDrawing()
svg = drawer.GetDrawingText()
with open("./paper_figs/DBAASP_3854.svg", "w") as f:
    f.write(svg)
# 绘制分子结构
img = Draw.MolToImage(mol, size=(900, 700))
# img.save('./paper_figs/DBAASP_3854.pdf', format="PDF")
display(img)

In [ ]:
!cairosvg ./paper_figs/DBAASP_3854.svg -o ./paper_figs/DBAASP_3854.pdf

#### 可视化添加了dummy atom之后的效果

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw


# 使用从 OPSIN 获得的 SMILES
smiles = 'CCC(C(=O)O)N'

# 创建分子对象
mol = Chem.MolFromSmiles(smiles)
drawer = Draw.MolDraw2DSVG(300, 300)
drawer.DrawMolecule(mol)
drawer.FinishDrawing()
svg = drawer.GetDrawingText()
with open("./paper_figs/AABA.svg", "w") as f:
    f.write(svg)

# 绘制分子结构
img = Draw.MolToImage(mol, size=(300, 300))  # , highlightAtoms=[4, 37]
# img.save('./paper_figs/AABA.pdf', format="PDF")
display(img)

In [ ]:
!cairosvg ./paper_figs/AABA.svg -o ./paper_figs/AABA.pdf

#### 画同一颜色的分子

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D


# 使用从 OPSIN 获得的 SMILES
smiles = 'C[C@H]([C@@H](C(=O)N[C@@H](CS)C(=O)O)NC(=O)[C@H](C(C)C)N)O'

# 创建分子对象
mol = Chem.MolFromSmiles(smiles)
drawer = Draw.MolDraw2DSVG(300, 300)
opts = drawer.drawOptions()
# new_palette = {6: (150/255, 135/255, 168/255), 8: (1, 1, 1)}
rdMolDraw2D.SetMonochromeMode(opts, (150/255, 135/255, 168/255), (1, 1, 1))
drawer.DrawMolecule(mol)
# opts.updateAtomPalette(new_palette)
drawer.DrawMolecule(mol)
drawer.FinishDrawing()
svg = drawer.GetDrawingText()
with open("./paper_figs/mol.svg", "w") as f:
    f.write(svg)

# 绘制分子结构
img = Draw.MolToImage(mol, size=(300, 300))  # , highlightAtoms=[4, 37]
# img.save('./paper_figs/AABA.pdf', format="PDF")
display(img)

In [ ]:
!cairosvg ./paper_figs/mol.svg -o ./paper_figs/mol.pdf

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D


# 使用从 OPSIN 获得的 SMILES
smiles = 'O=C(N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@@H]1C(=O)N[C@H](C(=O)N[C@@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)NCC1)[C@H](O)C)CCN)CCN)CC(C)C)CC(C)C)CCN)CCN)[C@H](O)C)CCN)CCCC(C)CC'

# 创建分子对象
mol = Chem.MolFromSmiles(smiles)
drawer = Draw.MolDraw2DSVG(300, 300)
opts = drawer.drawOptions()
# new_palette = {6: (150/255, 135/255, 168/255), 8: (1, 1, 1)}
rdMolDraw2D.SetMonochromeMode(opts, (150/255, 135/255, 168/255), (1, 1, 1))
drawer.DrawMolecule(mol)
# opts.updateAtomPalette(new_palette)
drawer.DrawMolecule(mol)
drawer.FinishDrawing()
svg = drawer.GetDrawingText()
with open("./paper_figs/mol_2.svg", "w") as f:
    f.write(svg)

# 绘制分子结构
img = Draw.MolToImage(mol, size=(1000, 1000))  # , highlightAtoms=[4, 37]
# img.save('./paper_figs/AABA.pdf', format="PDF")
display(img)

In [ ]:
!cairosvg ./paper_figs/mol_2.svg -o ./paper_figs/mol_2.pdf

#### 获得全部有PubChem cid的AMP的SMILES

In [ ]:
# 保存一个csv文件，DBAASP_id和对应的从PubChem直接得到的SMILES
import requests
import pandas as pd
import time

def PubChem_SMILES_to_df(data_smiles: list, DBAASP_id: int, url: str, cid: str, retries=3, delay=1) -> list:
    for i in range(retries):
        if i>0:
            print(f'retry: {i}')
        response = requests.get(url)
        if response.status_code == 200:
            smiles = response.text.strip()
            # 每一行保存三种信息，包括PubChem的cid
            data_smiles.append((DBAASP_id, smiles, cid))
            return data_smiles
        elif response.status_code == 503:
            print(f"Service unavailable (503) for DBAASP {DBAASP_id}. Retrying in", delay, "seconds...")
            time.sleep(delay)
    print(f"Failed to retrieve data of DBAASP {DBAASP_id}: response.status_code: ", response.status_code)
    

save_path = './Data/DBAASP_id_with_PubChem_SMILES.csv'
url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{}/property/IsomericSMILES/txt"
monomer_cnt = 0
monomer_w_cid_cnt = 0
multimer_cnt = 0
multimer_w_cid_cnt = 0
mulpep_cnt = 0
mulpep_w_cid_cnt = 0

# 创建一个空的 DataFrame，指定列名
id_smiles = []

for AMP in data:
    if AMP['complexity']['name'] == 'Monomer':
        monomer_cnt += 1
        if AMP.get('pubChemCid') is not None and AMP['pubChemCid'].get('cid') is not None:
            monomer_w_cid_cnt += 1
            id_smiles = PubChem_SMILES_to_df(id_smiles, AMP['id'], url.format(AMP['pubChemCid']['cid']), AMP['pubChemCid']['cid'])
    elif AMP['complexity']['name'] == 'Multimer':
        multimer_cnt += 1
        if AMP.get('pubChemCid') is not None and AMP['pubChemCid'].get('cid') is not None:
            multimer_w_cid_cnt += 1
            id_smiles = PubChem_SMILES_to_df(id_smiles, AMP['id'], url.format(AMP['pubChemCid']['cid']), AMP['pubChemCid']['cid'])
    elif AMP['complexity']['name'] == 'Multi-Peptide':
        mulpep_cnt += 1
        if AMP.get('pubChemCid') is not None and AMP['pubChemCid'].get('cid') is not None:
            mulpep_w_cid_cnt += 1
            id_smiles = PubChem_SMILES_to_df(id_smiles, AMP['id'], url.format(AMP['pubChemCid']['cid']), AMP['pubChemCid']['cid'])
print(f'{monomer_w_cid_cnt} / {monomer_cnt} Monomers have PubChem SMILES')
print(f'{multimer_w_cid_cnt} / {multimer_cnt} Multimers have PubChem SMILES')
print(f'{mulpep_w_cid_cnt} / {mulpep_cnt} Multi-Peptide have PubChem SMILES')
df = pd.DataFrame(id_smiles, columns=['DBAASP_id', 'SMILES', 'cid'])
df.to_csv(save_path, index=False)
        

上面结果可以看出来所有的有cid的peptide都有相应的smiles表示

#### 从PubChem获得所有的L- and D- Amino Acids的SMILES

In [ ]:
import requests
from typing import Tuple, List
import time
import json
import pandas as pd

AAs = [
    ("Alanine", "A"),
    ("Arginine", "R"),
    ("Asparagine", "N"),
    ("Aspartic acid", "D"),
    ("Cysteine", "C"),
    ("Glutamine", "Q"),
    ("Glutamic acid", "E"),
    ("Glycine", "G"),
    ("Histidine", "H"),
    ("Isoleucine", "I"),
    ("Leucine", "L"),
    ("Lysine", "K"),
    ("Methionine", "M"),
    ("Phenylalanine", "F"),
    ("Proline", "P"),
    ("Serine", "S"),
    ("Threonine", "T"),
    ("Tryptophan", "W"),
    ("Tyrosine", "Y"),
    ("Valine", "V")
]
url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{}/property/IsomericSMILES/JSON"
save_path='./Data/L-_and_D-Amino_Acid_SMILES.csv'

def get_AA_SMILES(url: str, AA_names: Tuple[str, str], retries=3, delay=1) -> List[List[str]]:
    """
    获得并返回氨基酸的SMILES
    :param url: url模版
    :param AA_names: (全称，大写单字母) 全称用于PubChem下载，单字母表示用于存储
    :return: smiles
    """
    iso_smiles = []
    for isoform_type in ['L-', 'D-']:
        for retry in range(retries):
            if retry>0:
                print(f'retry: {retry}')
            iso_AA_name = isoform_type + AA_names[0]
            response = requests.get(url.format(iso_AA_name))
            if response.status_code == 200:
                json_info = json.loads(response.text.strip())
                iso_smiles.append((iso_AA_name, AA_names[1] if isoform_type=='L-' else AA_names[1].lower(), json_info['PropertyTable']['Properties'][0]['IsomericSMILES']))
                break
            elif response.status_code == 503:
                print(f"Service unavailable (503) for AA: {iso_AA_name}, retrying in", delay, "seconds...")
                time.sleep(delay)
            else:
                print(f"Failed to retrieve data of AA: {iso_AA_name}: response.status_code: {response.status_code}")
                break
        if retry==retries-1:
            print(f"Failed to retrieve data of AA: {iso_AA_name}: response.status_code: {response.status_code}")
    return iso_smiles

aa_smils = []  
for aa in AAs:
    aa_smils.extend(get_AA_SMILES(url, aa))
df = pd.DataFrame(aa_smils, columns=['AA_full_name', 'AA_single_name', 'SMILES'])
df.to_csv(save_path, index=False)
for line in aa_smils:
    print(line)

上面这个不包含小写的g是因为Glycine没有异构体

#### 去除掉所有已经有PubChem cid的AMP，剩下的自行合成SMILES，这里先检查还有没有O
## 这里得到没有 PubChem cid 的 AMP 的列表
data_wo_cid_SMILES

In [ ]:
import pandas as pd


with_SMILES_path = './Data/DBAASP_id_with_PubChem_SMILES.csv'
df = pd.read_csv(with_SMILES_path)
DBAASP_ids_with_SMILES = df["DBAASP_id"].tolist()

# 获得没有现成SMILES的数据
data_wo_cid_SMILES = []
for AMP in data:
    if AMP['id'] not in DBAASP_ids_with_SMILES:
        data_wo_cid_SMILES.append(AMP)

# 把没有 PubChem SMILES 的数据存储到文件中方便读取
with open("./Data/peptides_wo_PubChem_SMILES_data.json", "w", encoding="utf-8") as json_file:
    json.dump(data_wo_cid_SMILES, json_file, ensure_ascii=False, indent=4)

# 直接从上面拿下来的代码
unique_characters = set()
unique_uppercase_characters = set()
unique_lowercase_characters = set()
for AMP in data_wo_cid_SMILES:
    if AMP['complexity']['name'] == 'Monomer' and AMP['sequence'] != None:
        for char in AMP['sequence']:
            unique_characters.add(char)
    else:
        for monomer in AMP['monomers']:
            for char in monomer['sequence']:
                unique_characters.add(char)
print("所有出现过的字符:", unique_characters)
for char in unique_characters:
    if char.isupper():
        unique_uppercase_characters.add(char)
    if char.islower():
        unique_lowercase_characters.add(char)
sorted_uppercase_characters = sorted(unique_uppercase_characters)
sorted_lowercase_characters = sorted(unique_lowercase_characters)
print(f"uppercase: {sorted_uppercase_characters}\tlength: {len(sorted_uppercase_characters)}")
print("lowercase:", sorted_lowercase_characters)

可以看到这上面已经没有O了

#### 检查没有 PubChem cid 的里面有多少是有 smiles 的

In [ ]:
no_PubChem_cid_w_smiles = []
for AMP in data_wo_cid_SMILES:
    if AMP['smiles'] is not None and len(AMP['smiles']) > 0:
        print(f"id:{AMP['id']}: {AMP['smiles'].strip()}")
        no_PubChem_cid_w_smiles.append(AMP)

print(f'num of AMPs without PubChem but has smiles: {len(no_PubChem_cid_w_smiles)}')
# print(json.dumps(no_PubChem_cid_w_smiles, ensure_ascii=False, indent=4))

#### 把这些有 smiles 的 AMP 也单独保存为 DBAASP_id 对 smiles 的 csv

In [ ]:
id_smiles = []
for AMP in no_PubChem_cid_w_smiles:
    id_smiles.append((AMP['id'], AMP['smiles'].strip()))

save_path = './Data/DBAASP_id_wo_PubChem_SMILES_w_DBAASP_smiles.csv'
df = pd.DataFrame(id_smiles, columns=['DBAASP_id', 'SMILES'])
df.to_csv(save_path, index=False)

#### 统计现在还有多少 X 和 x

In [ ]:
import re
count_X_x = 0
check_char = '[Xx]'
print(f'\n###############\nchecked character: "{check_char}"')
for AMP in data_wo_cid_SMILES:
    if AMP['complexity']['name'] == 'Monomer':
        if re.search(r"[Xx]", AMP['sequence']):
            print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence']},\tmonomer:{AMP['monomers']}')
            count_X_x += 1
    else:
        # print(1)
        for i, monomer in enumerate(AMP['monomers']):
            if re.search(r"[Xx]", monomer['sequence']):
                print(f'id:{AMP['id']},\ttype:{AMP['complexity']['name']},\tsequence:{AMP['sequence'] if AMP['sequence'] is not None else None},\tmonomer {i} sequence:{monomer['sequence']}')
                count_X_x += 1
print(f'X and x appeared in {count_X_x} sequences of data w/o cid SMILES')

原来有4858个

#### 检查没有 cid SMILES 的 AMPs 还有多少有 intrachain Bonds

In [ ]:
cycle_count = 0
intra_bond_DBAASP_id = []
for AMP in data_wo_cid_SMILES:
    if len(AMP.get('intrachainBonds', [])) > 0:
        intra_bond_DBAASP_id.append(AMP['id'])
        cycle_count += 1
print(intra_bond_DBAASP_id)
print(f'cycle count: {cycle_count}')

所以其实并没有因为PubChem减少多少，原来有4858个

#### 获得所有剩余intrachain Bond的种类

In [ ]:
type = []
type_id = []
cycleType = []
cycleType_id = []
chainParticipating = []
chainParticipating_id = []

for AMP in data_wo_cid_SMILES:
    intrachainBonds = AMP.get('intrachainBonds', [])
    if len(intrachainBonds) > 0:
        # print(AMP['id'])
        for bond in intrachainBonds:
            if bond['type']['id'] not in type_id:
                type_id.append(bond['type']['id'])
                type.append(bond['type'])
                
            # cycleType 不能用id识别重复内容，因为重复内容也可能id一样
            if bond['cycleType']['description'] not in cycleType_id:
                cycleType_id.append(bond['cycleType']['description'])
                cycleType.append(bond['cycleType'])
                
            if bond.get('chainParticipating', None) is not None:
                if bond['chainParticipating']['id'] not in chainParticipating_id:
                    chainParticipating_id.append(bond['chainParticipating']['id'])
                    chainParticipating.append(bond['chainParticipating'])

print(f'type: {len(type)}\n', json.dumps({'type':type}, indent=4))
print(f'cycleType: {len(cycleType)}\n', json.dumps(cycleType, indent=4))
print('chainParticipating: \n', json.dumps(chainParticipating, indent=4))

# 把这些可能的 bond 保存成 json 文件方便记录每一种 bond 的反应是怎么样的
save_path = 'Data/bonds_w_chinese_comment.json'
with open(save_path, "w", encoding="utf-8") as json_file:
    json.dump({'type':type, 'cycleType':cycleType, 'chainParticipating':chainParticipating}, json_file, ensure_ascii=False, indent=4)
print(f'\njson file saved to {save_path}')

#### 检查 Thioester Bond 是不是都是 Sidechain-Mainchain Bond 

In [ ]:
import json
SM = 0
MM = 0
SS = 0
Thieo = 0

ids = []

with open('./Data/processable_data_wo_cid_SMILES_new.json', 'r', encoding='utf-8') as file:
    processable_data_wo_cid_SMILES = json.load(file)

for AMP in processable_data_wo_cid_SMILES:
    intrachainBonds = AMP.get('intrachainBonds', [])
    if len(intrachainBonds) > 0:
        # print(AMP['id'])
        for bond in intrachainBonds:
            if bond['type']['name'] == 'CAR':#'(E)-but-2-enyl-B':# 'BisMeBn-B':
                Thieo += 1
                if bond.get('chainParticipating', None) is not None:
                    print(f"AMP id {AMP['id']}")
                    ids.append(AMP['id'])
                    print(json.dumps(bond, indent=4))
                    if bond['chainParticipating']['id'] == 2:
                        SS += 1
                    elif bond['chainParticipating']['id'] == 1:
                        MM += 1
                    else:
                        SM += 1
                        # print(json.dumps(AMP, indent=4))
                    
print(f'Thieo: {Thieo}')
print(f'Side-Side: {SS}')
print(f'Side-Main: {SM}')
print(f'Main-Main: {MM}')
print(f'ids:{ids}')
                    

#### 检查 cycleType 和 chain participating 的关系

In [ ]:
SM = 0
MM = 0
SS = 0
Thieo = 0

for AMP in data_wo_cid_SMILES:
    intrachainBonds = AMP.get('intrachainBonds', [])
    if len(intrachainBonds) > 0:
        # print(AMP['id'])
        for bond in intrachainBonds:
            if bond['cycleType']['description'] == 'Methyl lanthionine':
                Thieo += 1
                if bond.get('chainParticipating', None) is not None:
                    if bond['chainParticipating']['id'] == 2:
                        SS += 1
                    elif bond['chainParticipating']['id'] == 1:
                        MM += 1
                    else:
                        SM += 1
                    
print(f'Thieo: {Thieo}')
print(f'Side-Side: {SS}')
print(f'Side-Main: {SM}')
print(f'Main-Main: {MM}')

#### 检查剩余所有 interchainBonds 种类

In [ ]:
type = []
type_id = []
cycleType = []
cycleType_id = []
chainParticipating = []
chainParticipating_id = []

for AMP in data_wo_cid_SMILES:
    interchainBonds = AMP.get('interchainBonds', [])
    if len(interchainBonds) > 0:
        # print(AMP['id'])
        # print(json.dumps(interchainBonds, indent=4))
        # print(AMP['monomers'][0]['id'])
        # print(AMP['monomers'][0]['sequence'])
        # print(AMP['monomers'][1]['id'])
        # print(AMP['monomers'][1]['sequence'])
        for bond in interchainBonds:
            if bond['type']['name'] not in type_id:
                type_id.append(bond['type']['name'])
                type.append(bond['type'])

            # cycleType 不能用id识别重复内容，因为重复内容也可能id一样
            # if bond['cycleType']['description'] not in cycleType_id:
            #     cycleType_id.append(bond['cycleType']['description'])
            #     cycleType.append(bond['cycleType'])

            if bond.get('chainParticipating', None) is not None:
                if bond['chainParticipating']['id'] not in chainParticipating_id:
                    chainParticipating_id.append(bond['chainParticipating']['id'])
                    chainParticipating.append(bond['chainParticipating'])

print(f'type: {len(type)}\n', json.dumps({'type':type}, indent=4))
print(f'cycleType: {len(cycleType)}\n', json.dumps(cycleType, indent=4))
print('chainParticipating: \n', json.dumps(chainParticipating, indent=4))

# 把这些可能的 bond 保存成 json 文件方便记录每一种 bond 的反应是怎么样的
save_path = 'Data/bonds_w_chinese_comment_inter.json'
with open(save_path, "w", encoding="utf-8") as json_file:
    json.dump({'type':type, 'cycleType':cycleType, 'chainParticipating':chainParticipating}, json_file, ensure_ascii=False, indent=4)
print(f'\njson file saved to {save_path}')

#### 检查所有的 Multimer 有没有其中某个单链不属于 DBAASP

In [ ]:
num_no_DBAASP_id = 0
for AMP in data_wo_cid_SMILES:
    if AMP['complexity']['name'] == 'Multimer' or AMP['complexity']['name'] == 'Multi-Peptide':
        for seq in AMP['monomers']:
            if seq['dbaaspId'] is None or len(seq['dbaaspId']) == 0:
                num_no_DBAASP_id += 1
print(f'num_no_DBAASP_id: {num_no_DBAASP_id}')

所以所有的Multimer或者Multi-Peptide的单体都是DBAASP中的一条序列，可以按照相同的格式操作

#### 搜集所有的 unusualAminoAcids
只用处理 Monomer 的就行了，因为可见所有的Multimer或者Multi-Peptide的单体都是DBAASP中的一条序列

In [ ]:
unusual_aas = []
names = []
for AMP in data_wo_cid_SMILES:
    if AMP['complexity']['name'] == 'Monomer':
        if len(AMP['unusualAminoAcids']) > 0:
            for unusual_aa in AMP['unusualAminoAcids']:
                if unusual_aa['modificationType']['name'] not in names:
                    names.append(unusual_aa['modificationType']['name'])
                    unusual_aas.append({unusual_aa['modificationType']['name']:unusual_aa['modificationType']['description']})

# 存储内容到文件
with open("./Data/unusual_aa_names.json", "w", encoding="utf-8") as json_file:
    json.dump(unusual_aas, json_file, ensure_ascii=False, indent=4)

print(f'num of kinds of unusual amino acids: {len(unusual_aas)}')

#### 过滤所有能从PubChem得到的 unusual amino acid 的 smiles

In [ ]:
import re
import json
import pandas as pd
from typing import List
import requests
import time

url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{}/property/IsomericSMILES/JSON"
save_path='./Data/unusual_aa_smiles_from_PubChem.csv'

# 打开并读取JSON文件
with open("./Data/unusual_aa_names.json", 'r', encoding='utf-8') as file:
    unusual_aas = json.load(file)  # 将JSON内容加载为Python列表

def get_unusual_AA_SMILES(url: str, names, retries=3, delay=1) -> List[List[str]]:
    """
    获得并返回 unusual 氨基酸的SMILES
    :param url: url模版
    :param names: 名称用于PubChem下载
    :return: smiles
    """
    smiles = None
    for name in names:
        for retry in range(retries):
            if retry>0:
                print(f'retry: {retry}')
            response = requests.get(url.format(name))
            if response.status_code == 200:
                json_info = json.loads(response.text.strip())
                smiles = json_info['PropertyTable']['Properties'][0]['IsomericSMILES']
                break
            elif response.status_code == 503:
                print(f"Service unavailable (503) for name: {name}, retrying in", delay, "seconds...")
                time.sleep(delay)
            else:
                print(f"Failed to retrieve data of name: {name}: response.status_code: {response.status_code}")
                break
        if retry==retries-1:
            print(f"Failed to retrieve data of name: {name}: response.status_code: {response.status_code}")
        if smiles is not None:
            break
    return name, smiles

no_PubChem_smiles = []
have_PubChem_smiles = []

for unusual_aa in unusual_aas:
    names = []
    names.append(list(unusual_aa.keys())[0])
    split_results = re.split(r'[;,]\s+', list(unusual_aa.values())[0])

    # 使用列表推导式过滤掉以 C 紧接着数字开头的字符串：C7H16N4O2
    filtered_results = [s for s in split_results if not re.match(r'^C\d+', s)]
    names.extend(filtered_results)

    # 这里返回的 unusual_name 是真正从 PubChem 得到 smiles 的名字，而不是DBAASP里面记录的name
    unusual_name, smiles = get_unusual_AA_SMILES(url, names)
    if smiles is not None:
        have_PubChem_smiles.append((names[0], smiles))
        # have_PubChem_smiles.append((unusual_name, smiles))
    else:
        no_PubChem_smiles.append(unusual_aa)

df = pd.DataFrame(have_PubChem_smiles, columns=['unusual_AA', 'SMILES'])
# df.to_csv('./Data/unusual_aa_w_PubChem_smiles.csv', index=False)
df.to_csv('./Data/unusual_aa_w_PubChem_smiles_new.csv', index=False)

# with open("./Data/unusual_aa_names_wo_PubChem_smiles.json", "w", encoding="utf-8") as json_file:
with open("./Data/unusual_aa_names_wo_PubChem_smiles_new.json", "w", encoding="utf-8") as json_file:
    json.dump(no_PubChem_smiles, json_file, ensure_ascii=False, indent=4)

print(f'num of unusual aas with PubChem smiles: {len(have_PubChem_smiles)}')
print(f'num of unusual aas without PubChem smiles: {len(no_PubChem_smiles)}')

#### 比较新旧 unusual_aa_w_PubChem_smiles 的差异名称，就能知道旧的比新的多出来的是之前有错的

In [ ]:
import pandas as pd

unusual_aas_w_PubChem_smiles_old_file_path = './Data/unusual_aa_w_PubChem_smiles.csv'
unusual_aas_w_PubChem_smiles_new_file_path = './Data/unusual_aa_w_PubChem_smiles_new.csv'

df_old = pd.read_csv(unusual_aas_w_PubChem_smiles_old_file_path)
df_new = pd.read_csv(unusual_aas_w_PubChem_smiles_new_file_path)

old_names = df_old['unusual_AA'].tolist()
new_names = df_new['unusual_AA'].tolist()

wrong_ones = list(set(old_names) - set(new_names))
what_is_this = list(set(new_names) - set(old_names))

print('wrong', wrong_ones)
print(f'length of wrong ones: {len(wrong_ones)}')
print(f'set length of wrong ones: {len(set(wrong_ones))}')
print('no idea', what_is_this)

#### 从原始的 unusual_aa_names.json 里面提取对应的表述，并且去掉分子量表达式

In [ ]:
import json
import re

with open('./Data/unusual_aa_names.json', 'r', encoding='utf-8') as file:
    unusual_aas_desc = json.load(file)  # 将JSON内容加载为Python列表

lines = []
for unusual_aa_desc in unusual_aas_desc:
    if list(unusual_aa_desc.keys())[0] in wrong_ones:
        split_results = re.split(r'[;,]\s+', list(unusual_aa_desc.values())[0])
        # 使用列表推导式过滤掉以 C 紧接着数字开头的字符串：C7H16N4O2
        filtered_results = [s for s in split_results if not re.match(r'^C\d+', s)]
        line = ', '.join(filtered_results)
        lines.append(list(unusual_aa_desc.keys())[0]+'. '+line)

# 将列表内容逐行写入文本文件
with open("./Data/wrong_ones_desc.txt", "w") as file:
    for line in lines:
        file.write(line + "\n")  # 每行末尾添加换行符

#### 错误的部分重新利用 OPSIN 生成正确的 SMILES
wrong_ones_desc_refined.txt是利用GPT优化过的IUPAC表达

In [ ]:
!java -jar /home/tianang/opsin-cli-2.8.0-jar-with-dependencies.jar -osmi ./Data/wrong_ones_desc_refined.txt ./Data/wrong_ones_OPSIN_output.txt

#### 从 wrong_ones_desc.txt 读取这些正确 OPSIN 生成的 SMILES 所对应的名字并替换到 all_aa_smiles.txt 里

In [ ]:
import pandas as pd
import copy

# 有name的，', '前面的就是对应的 name
wrong_ones_desc_path = './Data/wrong_ones_desc.txt'
wrong_names = []

# 读取错误smiles对应的name
with open(wrong_ones_desc_path, 'r', encoding='utf-8') as file:
    while True:
        line = file.readline()
        if not line:  # 文件结束时跳出循环
            break
        wrong_names.append(line.strip().split('.')[0])
# print(names, len(names))

# 读取错误smiles的正确版本
wrong_ones_OPSIN_smiles_path = './Data/wrong_ones_OPSIN_output.txt'
correct_smiles = []
with open(wrong_ones_OPSIN_smiles_path, 'r', encoding='utf-8') as file:
    while True:
        line = file.readline()
        if not line:  # 文件结束时跳出循环
            break
        correct_smiles.append(line.strip())

# 读取之前存在错误smiles的全部all_aa_smiles.csv
df = pd.read_csv('./Data/all_aa_smiles.csv')
old_all_aa_smiles = df.values.tolist()
new_all_aa_smiles = copy.deepcopy(old_all_aa_smiles)
for line_idx, old_aa_smiles in enumerate(old_all_aa_smiles):
    for wrong_name, correct_smile in zip(wrong_names, correct_smiles):
        if old_aa_smiles[0] == wrong_name:
            new_all_aa_smiles[line_idx] = [wrong_name, correct_smile]

df_new = pd.DataFrame(new_all_aa_smiles, columns=['aa', 'SMILES'])
df_new.to_csv('./Data/all_aa_smiles_new.csv', index=False, columns=['aa', 'SMILES'])



#### 把没有 PubChem SMILES 的 unusual aa 的描述存入 text 文件给 GPT 为 IUPAC 命名

In [ ]:
import json

# 打开并读取JSON文件
with open('./Data/unusual_aa_names_wo_PubChem_smiles.json', 'r', encoding='utf-8') as file:
    no_PubChem_smiles_unusual_aas = json.load(file)  # 将JSON内容加载为Python列表

lines = []
for no_PubChem_smiles_unusual_aa in no_PubChem_smiles_unusual_aas:
    lines.append(list(no_PubChem_smiles_unusual_aa.values())[0])

# 将列表内容逐行写入文本文件
with open("./Data/unusual_aa_text_not_transfered_by_GPT.txt", "w") as file:
    for line in lines:
        file.write(line + "\n")  # 每行末尾添加换行符

#### 把这个保存好的 text 文件中的内容送给GPT-o1输出标准的 IUPAC 命名
保存到 unusual_aa_text_transfered_by_GPT.txt

#### unusual_aa_text_transfered_by_GPT.txt 送到 OPSIN 里转换成 SMILES

In [ ]:
!java -jar /home/tianang/opsin-cli-2.8.0-jar-with-dependencies.jar -osmi ./Data/unusual_aa_text_transfered_by_GPT.txt ./Data/unusual_aa_smiles_OPSIN_output.txt

#### OPSIN 转换好的 SMILES 重新 map 到有对应 unusual aa 的
文件存储在 unusual_aa_wo_PubChem_smiles.csv

In [ ]:
import json
import pandas as pd
OPSIN_smiles = []
name_smiels = []

# 读取 OPSIN 处理好的 SMILES
with open('./Data/unusual_aa_smiles_OPSIN_output.txt', 'r', encoding='utf-8') as file:
    for line in file:
        # 去掉行尾的换行符（可选）
        line = line.strip()
        # 打印每一行（或者处理数据）
        OPSIN_smiles.append(line)

# 打开并读取JSON文件
with open('./Data/unusual_aa_names_wo_PubChem_smiles.json', 'r', encoding='utf-8') as file:
    no_PubChem_smiles_unusual_aas = json.load(file)  # 将JSON内容加载为Python列表

for no_PubChem_smiles_unusual_aa, smiles in zip(no_PubChem_smiles_unusual_aas, OPSIN_smiles):
    name = list(no_PubChem_smiles_unusual_aa.keys())[0]
    name_smiels.append((name, smiles))

df = pd.DataFrame(name_smiels, columns=['unusual_AA', 'SMILES'])
df.to_csv('./Data/unusual_aa_wo_PubChem_smiles.csv', index=False)

#### 把所有包含 命名-SMILES 映射信息的文件合并到 all_aa_smiles.csv
三个文件分别是： <br>
L-_and_D-Amino_Acid_SMILES.csv <br>
unusual_aa_w_PubChem_smiles.csv <br>
unusual_aa_wo_PubChem_smiles.csv

In [ ]:
import pandas as pd

# 重命名的列名
universal_colume_name = ['aa', 'SMILES']

# 读取第一个有多列的 CSV 文件，并提取两列
L_and_D_Amino_Acid_SMILES = "./Data/L-_and_D-Amino_Acid_SMILES.csv"
df1 = pd.read_csv(L_and_D_Amino_Acid_SMILES)  # 读取文件
columns_to_extract = ['AA_single_name', 'SMILES']  # 替换为你需要的列名
df1_extracted = df1[columns_to_extract]  # 提取指定列

# 重命名列名
df1_extracted.columns = universal_colume_name

# 读取第二个只有两列的 CSV 文件
unusual_aa_w_PubChem_smiles = "./Data/unusual_aa_w_PubChem_smiles.csv"
df2 = pd.read_csv(unusual_aa_w_PubChem_smiles)  # 读取文件

# 重命名列名
df2.columns = universal_colume_name

unusual_aa_wo_PubChem_smiles = "./Data/unusual_aa_wo_PubChem_smiles.csv"
df3 = pd.read_csv(unusual_aa_wo_PubChem_smiles)  # 读取文件

# 重命名列名
df3.columns = universal_colume_name

# 拼接两个 DataFrame（按行拼接）
result = pd.concat([df1_extracted, df2, df3], ignore_index=True)

# 保存结果到新文件（可选）
result.to_csv("./Data/all_aa_smiles.csv", index=False)

print("拼接完成，保存为 all_aa_smiles.csv")

### 注意这上面最后使用的并不是 all_aa_smiles.csv, 而是手动修改过的更准确的 all_aa_smiles_handcrafted.csv
#### 检查每一个氨基酸是不是都能找到 N 和 C，记录输出那些找不到 N C terminal 的

In [ ]:
import pandas as pd
from aa_seq_to_smiles import *

df = pd.read_csv('./Data/all_aa_smiles_new.csv')

# 变成 list 格式，[name, smiles]
all_aa_smiles = df.values.tolist()

count = 0
for name, smiles in all_aa_smiles:
    if name=='AGL':
        print(f'{name}: {smiles}')
    aa = AAs(smiles, name)
    if aa.N_terminal_atom is None or aa.C_terminal_atom is None:
        count += 1

print(f'\nnum of aas that can\'t find N,C terminals: {count}')

#### 检查 PFPh 这个 unusual 氨基酸出现在哪些 AMP 中
记录下来不处理这些 peptide

In [ ]:
from tqdm import tqdm
import json

PFPh_peptides = []

for AMP in data:
    if AMP['complexity']['name'] == 'Monomer':
        for unusual_aa in AMP['unusualAminoAcids']:
            if unusual_aa['modificationType']['name'] == 'PFPh':
                print('\nAMP id: ', AMP['id'])
                print('AMP seq: ', AMP['sequence'])
                print('unusual_aa')
                print(json.dumps(unusual_aa, indent=4, ensure_ascii=False))
                PFPh_peptides.append(AMP['id'])

print(PFPh_peptides)

#### 检查所有的只能找到 N 端或者只能找到 C 端的氨基酸是不是只出现在边边上
#### 把不合格的 Monomer 的 peptides id 记录下来

In [ ]:
no_C_unusual_aas = ['DAB', 'Dmp', 'DHL', 'BDZ', 'Npm', 'Pip']
no_N_unusual_aas = ['Phg', 'DHA', 'LAP', 'Iva', 'Piz', 'Epa', 'Api', 'ARGol', 'OBU']

peptide_count = 0

un_processable_peptides = []

for unusual_aas in [no_C_unusual_aas, no_N_unusual_aas]:
    for abn_unusual_aa in unusual_aas:
        if abn_unusual_aa in no_C_unusual_aas:
            print(f'\n{abn_unusual_aa} should be on position end')
        else:
            print(f'\n{abn_unusual_aa} should be on position 1')
        for AMP in data:
            if AMP['complexity']['name'] == 'Monomer':
                for unusual_aa in AMP['unusualAminoAcids']:
                    if unusual_aa['modificationType']['name'] == abn_unusual_aa:
                        # peptide_count += 1
                        if abn_unusual_aa in no_C_unusual_aas:
                            if len(AMP['sequence']) == unusual_aa['position']:
                                pass
                                print(f'\n{AMP['id']} on the right position')
                                print('seq: ', AMP['sequence'])
                            else:
                                peptide_count += 1
                                un_processable_peptides.append(AMP['id'])
                                print(f'\n{AMP['id']} on the WRONG position, should be at the end')
                                print('seq: ', AMP['sequence'])
                            print(json.dumps(unusual_aa, indent=4, ensure_ascii=False))
                        else:
                            if unusual_aa['position'] == 1:
                                pass
                                print(f'\n{AMP['id']} on the right position')
                                print('seq: ', AMP['sequence'])
                            else:
                                peptide_count += 1
                                un_processable_peptides.append(AMP['id'])
                                print(f'\n{AMP['id']} on the WRONG position, should be on 1')
                                print('seq: ', AMP['sequence'])
                            print(json.dumps(unusual_aa, indent=4, ensure_ascii=False))

un_processable_peptides = list(set(un_processable_peptides))
print('\nnum of peptides with abnormal aas: ', len(un_processable_peptides))

#### 去掉那些不合格的 peptides, 包括 Multimer 和 Multi-peptide
不合格 peptides 的 id 在上面 unprocessable_peptides 和 PFPh_peptides 中 <br>
暂时可以处理的 peptides 存储在 processable_data_wo_cid_SMILES

In [ ]:
processable_data_wo_cid_SMILES = []

# 混合包含没有 N，C 端的和两个都没有的 PFPh 的所有 peptides
un_processable_peptides.extend(PFPh_peptides)

print('Creating processable_data_wo_cid_SMILES')

for AMP in data_wo_cid_SMILES:
    if AMP['complexity']['name'] == 'Monomer':
        if AMP['id'] not in un_processable_peptides:
            processable_data_wo_cid_SMILES.append(AMP)
    else:
        process_flag = True
        for monomer in AMP['monomers']:
            if monomer['id'] in un_processable_peptides:
                process_flag = False
        if process_flag:
            processable_data_wo_cid_SMILES.append(AMP)

print('Saving file ...')
# 把这些可以处理的没有 PubChem cid SMILES 的存到文件中方便读取
with open("./Data/processable_data_wo_cid_SMILES.json", "w", encoding="utf-8") as json_file:
    json.dump(processable_data_wo_cid_SMILES, json_file, ensure_ascii=False, indent=4)

print('num of processable AMPs without PubChem SMILES: ', len(processable_data_wo_cid_SMILES))
print('num of AMPs without PubChem SMILES: ', len(data_wo_cid_SMILES))

#### 测试 Peptide 类在所有 processable AMPs 上的 mainchain 连接效果

In [ ]:
from aa_seq_to_smiles import *
from tqdm import tqdm

aa_smiles_dict = get_aa_smiles_dict('./Data/all_aa_smiles_new.csv')

for AMP in tqdm(processable_data_wo_cid_SMILES, desc='Processing  AMPs'):
    if AMP['complexity']['name'] == 'Monomer':
        AMP_peptide = Peptide(AMP['sequence'], aa_smiles_dict, AMP['id'], AMP['intrachainBonds'], AMP['interchainBonds'], AMP['unusualAminoAcids'])
    elif AMP['complexity']['name'] == 'Multimer':
        seqs = []
        multimer_id = AMP['id']
        intrachain_bonds = []
        interchain_bonds = []
        unusual_aas = []
        for monomer in AMP['monomers']:
            seqs.append(monomer['sequence'])
            intrachain_bonds.append(monomer['intrachainBonds'])
            interchain_bonds.append(monomer['interchainBonds'])
            unusual_aas.append(monomer['unusualAminoAcids'])
        AMP_peptide = Peptide(seqs, aa_smiles_dict, multimer_id, intrachain_bonds, interchain_bonds, unusual_aas)

#### 修改 DBAASP 中有个别有问题的数据

In [ ]:
import json
# 读取 DBAASP 可处理数据的代码，这部分应该变成一个函数
with open('./Data/processable_data_wo_cid_SMILES.json', 'r', encoding='utf-8') as file:
    processable_data_wo_cid_SMILES = json.load(file)

for AMP in processable_data_wo_cid_SMILES:
    if AMP['id'] == 21676:
        for bond in AMP['intrachainBonds']:
            if bond['type']['name'] == 'EST' and bond['position1']==0:
                bond['position1']=2
                # bond['chainParticipating']['name'] = 'MMB'
                # bond['chainParticipating']['description'] = 'Mainchain-Mainchain Bond'
        # for unusual_aa in AMP['unusualAminoAcids']:
        #     if unusual_aa['position']==18:
        #         unusual_aa['position']=26

# 保存回 JSON 文件
with open('./Data/processable_data_wo_cid_SMILES.json', 'w', encoding='utf-8') as file:
    json.dump(processable_data_wo_cid_SMILES, file, ensure_ascii=False, indent=4)

print("修改已保存！")

#### 去掉 ./Data/processable_data_wo_cid_SMILES.json 中才发现的那些没有 PubChem smiles 但是有 DBAASP smiles 的 AMP

In [ ]:
import pandas as pd
df = pd.read_csv('./Data/DBAASP_id_wo_PubChem_SMILES_w_DBAASP_smiles.csv')

# 假设想要获取列名为 "column_name" 的这一列数据，并转换成列表
DBAASP_id_list_with_new_smiles = df['DBAASP_id'].tolist()

with open('./Data/processable_data_wo_cid_SMILES.json', 'r', encoding='utf-8') as file:
    processable_data_wo_cid_SMILES = json.load(file)

new_processable_data_wo_cid_SMILES = []

for AMP in processable_data_wo_cid_SMILES:
    if AMP['id'] not in DBAASP_id_list_with_new_smiles:
        new_processable_data_wo_cid_SMILES.append(AMP)

# 保存回 JSON 文件
with open('./Data/processable_data_wo_cid_SMILES_new.json', 'w', encoding='utf-8') as file:
    json.dump(new_processable_data_wo_cid_SMILES, file, ensure_ascii=False, indent=4)

print("数据保存在 ./Data/processable_data_wo_cid_SMILES_new.json ！")

#### 检查 valency 的代码

In [ ]:
from rdkit import Chem

# 创建一个分子对象（以乙烯为例：C=CH2）
mol = Chem.MolFromSmiles("NCC(=O)O")

# 遍历分子中的原子并获取显式电子配位数
for atom in mol.GetAtoms():
    atom_idx = atom.GetIdx()  # 原子索引
    atomic_symbol = atom.GetSymbol()  # 原子符号
    explicit_valence = atom.GetExplicitValence()  # 显式电子配位数

    print(f"Atom {atom_idx} ({atomic_symbol}): Explicit Valence = {explicit_valence}")

#### 一点 e3nn 的尝试

In [ ]:
import e3nn.o3 as o3
irreps_input = o3.Irreps("10x0e + 5x1o + 2x2e")
irreps_query = o3.Irreps("11x0e + 4x1o")
print(o3.Linear(irreps_input, irreps_query))
print('130 = 11 x 10 + 5 x 4')
o3.FullTensorProduct("1x2e", "1x3o").visualize();
# o3.Linear(irreps_input, irreps_query).visualize();

#### 先看一下 N_terminal 和 C_terminal 的修饰什么情况

In [ ]:
import re
import pandas as pd

cTerminus_name = []
nTerminus_name = []
cTerminus_csv_data = []
nTerminus_csv_data = []
cTerminus_modification = []
nTerminus_modification = []
num_pep_count = 0

for AMP in data:
    cTerminus = AMP.get('cTerminus')  # Fetch `cTerminus` once
    if cTerminus and cTerminus.get('name') is not None:
        num_pep_count += 1
        if cTerminus.get('name') not in cTerminus_name:
            cTerminus_name.append(AMP.get('cTerminus', {}).get('name'))
            cTerminus_modification.append(AMP['cTerminus'])
            cTerminus_modification[-1]['count'] = 1
        else:
            cTerminus_modification[cTerminus_name.index(cTerminus.get('name'))]['count'] += 1
    nTerminus = AMP.get('nTerminus')  # Fetch `cTerminus` once
    if nTerminus and nTerminus.get('name') is not None:
        num_pep_count += 1
        if nTerminus.get('name') not in nTerminus_name:
            nTerminus_name.append(AMP.get('nTerminus', {}).get('name'))
            nTerminus_modification.append(AMP['nTerminus'])
            nTerminus_modification[-1]['count'] = 1
        else:
            nTerminus_modification[nTerminus_name.index(nTerminus.get('name'))]['count'] += 1

for cTerminal_modifi in cTerminus_modification:
    desc = re.split(r'[;,]\s+', cTerminal_modifi['description'])[0]s
    desc = desc.split('[C')[0].strip()
    cTerminus_csv_data.append([cTerminal_modifi['name'], desc])

for nTerminal_modifi in nTerminus_modification:
    desc = re.split(r'[;,]\s+', nTerminal_modifi['description'])[0]
    desc = desc.split('[C')[0].strip()
    nTerminus_csv_data.append([nTerminal_modifi['name'], desc])

print(f'num of peptides with c_n_terminus modification: {num_pep_count}')
print(f'cTerminus: length: {len(cTerminus_modification)}\n {json.dumps(cTerminus_modification, indent=4, ensure_ascii=False)}')
print(f'nTerminus: length: {len(nTerminus_modification)}\n {json.dumps(nTerminus_modification, indent=4, ensure_ascii=False)}')
print(f'cTerminus csv data:\n {json.dumps(cTerminus_csv_data, indent=4, ensure_ascii=False)}')
print(f'nTerminus csv data:\n {json.dumps(nTerminus_csv_data, indent=4, ensure_ascii=False)}')

c_data_df = pd.DataFrame(cTerminus_csv_data, columns=['name', 'SMILES'])
n_data_df = pd.DataFrame(nTerminus_csv_data, columns=['name', 'SMILES'])

c_data_df.to_csv('./Data/terminal_modifications/terminal_modification_c_origin.csv', index=False)
n_data_df.to_csv('./Data/terminal_modifications/terminal_modification_n_origin.csv', index=False)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw


# 使用从 OPSIN 获得的 SMILES
smiles = "CC(C)C1C(=O)O[C@H](C)[C@H](N)C(=O)N[C@H](C(C)C)C(=O)N2(C)CCC[C@H]2C(=O)N(C)CC(=O)N1C"

# 创建分子对象
mol = Chem.MolFromSmiles(smiles)

# 绘制分子结构
img = Draw.MolToImage(mol, size=(1000, 1000))
display(img)

#### 获取所有能从 PubChem 得到的 terminal modification 的 SMILES
这里需要手动切换 c_ n_ terminal

In [ ]:
import re
import json
import pandas as pd
from typing import List
import requests
import time

url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{}/property/IsomericSMILES/JSON"
# save_path='./Data/terminal_modifications/c_terminal_modification_smiles_from_PubChem.csv'

def get_unusual_AA_SMILES(url: str, names, retries=3, delay=1) -> List[List[str]]:
    """
    获得并返回 unusual 氨基酸的SMILES
    :param url: url模版
    :param names: 名称用于PubChem下载
    :return: smiles
    """
    smiles = None
    for name in names:
        for retry in range(retries):
            if retry>0:
                print(f'retry: {retry}')
            response = requests.get(url.format(name))
            if response.status_code == 200:
                json_info = json.loads(response.text.strip())
                smiles = json_info['PropertyTable']['Properties'][0]['IsomericSMILES']
                break
            elif response.status_code == 503:
                print(f"Service unavailable (503) for name: {name}, retrying in", delay, "seconds...")
                time.sleep(delay)
            else:
                print(f"Failed to retrieve data of name: {name}: response.status_code: {response.status_code}")
                break
        if retry==retries-1:
            print(f"Failed to retrieve data of name: {name}: response.status_code: {response.status_code}")
        if smiles is not None:
            break
    return name, smiles

no_PubChem_smiles = []
have_PubChem_smiles = []
c_name_smiles = []
n_name_smiles = []

# DBAASP 的命名和真正的化学名称
for DBAASP_name, name in nTerminus_csv_data:

    # 这里返回的 unusual_name 是真正从 PubChem 得到 smiles 的名字，而不是DBAASP里面记录的name
    _, smiles = get_unusual_AA_SMILES(url, [name])
    if smiles is not None:
        n_name_smiles.append([DBAASP_name, smiles])
        # have_PubChem_smiles.append((unusual_name, smiles))
    else:
        n_name_smiles.append([DBAASP_name, ' '])

df = pd.DataFrame(n_name_smiles, columns=['name', 'SMILES'])
# df.to_csv('./Data/unusual_aa_w_PubChem_smiles.csv', index=False)
df.to_csv('./Data/terminal_modifications/n_terminal_smiles_from_PubChem.csv', index=False)

#### 合并 PubChem 得到的完整 SMILES 和 DBAASP 自带的 SMILES 以及我们拼接的 SMILES 的 csv 文件

In [ ]:
import pandas as pd

# 定义三个CSV文件的路径
file1_path = "./Data/DBAASP_id_wo_existing_smiles_intra_linked_smiles.csv"
file2_path = "./Data/DBAASP_id_wo_PubChem_SMILES_w_DBAASP_smiles.csv"
file3_path = "./Data/DBAASP_id_with_PubChem_SMILES.csv"

# 读取三个CSV文件
df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)
df3 = pd.read_csv(file3_path)

# 合并三个文件，df3只保留DBAASP_id和SMILES列
df_combined = pd.concat([df1, df2, df3[['DBAASP_id', 'SMILES']]])

# 按DBAASP_id列从小到大排序
df_combined_sorted = df_combined.sort_values(by='DBAASP_id')

# 将合并后的结果保存到一个新的CSV文件
output_path = "./Data/DBAASP_id_SMILES_merged.csv"
df_combined_sorted.to_csv(output_path, index=False)
print(len(df_combined_sorted))
print(f"合并后的文件已保存到: {output_path}")

#### 输出不合格的 SMILES

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem

# 读取CSV文件
df = pd.read_csv('./Data/DBAASP_id_SMILES_merged.csv')

# 过滤掉mol为None的行
valid_rows = []
for index, row in df.iterrows():
    smiles = row['SMILES']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f'Invalid SMILES: {row['DBAASP_id']} | {smiles}')
    else:
        valid_rows.append(row)

# 将过滤后的数据转换为DataFrame
filtered_df = pd.DataFrame(valid_rows)

# 将更新后的DataFrame写回原CSV文件
filtered_df.to_csv('./Data/DBAASP_id_SMILES_merged.csv', index=False)

#### 看看有哪些细菌在所有的数据中都出现过

In [ ]:
from tqdm import tqdm

bact_list = []
for AMP in tqdm(data):
    bact_names = []
    if AMP['targetActivities'] is not None:
        for bact in AMP['targetActivities']:
            if bact['targetSpecies'] is not None:
                bact_names.append(bact['targetSpecies']['name'])
            else:
                print(AMP['id'])
                continue
        bact_list.append(set(bact_names))
    else:
        print(AMP['id'])

from functools import reduce
result = reduce(lambda x, y: x & y, bact_list)

print(result)

#### 从 bacteria_get.py 查看哪些菌株数据比较多, 这里查看比较多的菌株的 MIC 和 单位的情况
"Escherichia coli ATCC 25922": 7076, <br>
"Pseudomonas aeruginosa ATCC 27853": 4585, <br>
"Staphylococcus aureus ATCC 25923": 4430, <br>
"Staphylococcus aureus": 2516, <br>
"Staphylococcus aureus ATCC 29213": 2452, <br>
"Escherichia coli": 1949, <br>
"Pseudomonas aeruginosa": 1597, <br>
"Pseudomonas aeruginosa PA01": 1539, <br>
"Enterococcus faecalis ATCC 29212": 1524, <br>
"Acinetobacter baumannii ATCC 19606": 1463, <br>
"Staphylococcus epidermidis ATCC 12228: 1417, <br>
"Candida albicans ATCC 10231": 1206, <br>
"Klebsiella pneumoniae ATCC 700603": 1175, <br>
"Staphylococcus aureus ATCC 43300": 1110, <br>
"Salmonella enterica subsp. enterica serovar Typhimurium ATCC 14028": 1022, <br>
"Staphylococcus aureus ATCC 6538": 972, <br>
"Pseudomonas aeruginosa ATCC 9027": 963, <br>
"Candida albicans": 950, <br>
"Klebsiella pneumoniae": 924

In [ ]:
from tqdm import tqdm
# for huge_bact_name in ['Pseudomonas aeruginosa PAO1']:
for huge_bact_name in ["Escherichia coli ATCC 25922", "Pseudomonas aeruginosa ATCC 27853", "Staphylococcus aureus ATCC 25923", "Staphylococcus aureus", "Staphylococcus aureus ATCC 29213", "Escherichia coli", "Pseudomonas aeruginosa", "Pseudomonas aeruginosa PAO1", "Enterococcus faecalis ATCC 29212", "Acinetobacter baumannii ATCC 19606", "Staphylococcus epidermidis ATCC 12228", "Candida albicans ATCC 10231", "Klebsiella pneumoniae ATCC 700603", "Staphylococcus aureus ATCC 43300", "Salmonella enterica subsp. enterica serovar Typhimurium ATCC 14028", "Staphylococcus aureus ATCC 6538", "Pseudomonas aeruginosa ATCC 9027", "Candida albicans", "Klebsiella pneumoniae"]:
    measure_unit = {}
    for AMP in tqdm(data):
        bact_names = []
        bact_names_2 = []


        if AMP['targetActivities'] is not None:
            for bact in AMP['targetActivities']:

                if bact['targetSpecies'] is not None and bact['targetSpecies']['name'] == huge_bact_name:
                    # print(AMP['id'], 1)
                    # if bact['targetSpecies']['name'] not in list(bact_count.keys()):
                    #     bact_count[bact['targetSpecies']['name']] = 1
                    # else:
                    #     bact_count[bact['targetSpecies']['name']] += 1
                    if bact['unit'] is not None:
                        if (bact['activityMeasureValue'], bact['unit']['name']) not in list(measure_unit.keys()):
                            measure_unit[(bact['activityMeasureValue'], bact['unit']['name'])] = 1
                        else:
                            measure_unit[(bact['activityMeasureValue'], bact['unit']['name'])] += 1
                        # if (bact['activityMeasureValue'], bact['unit']['name']) not in measure_unit:
                        #     measure_unit.append((bact['activityMeasureValue'], bact['unit']['name']))
                    bact_names.append(bact['targetSpecies']['name'])
                    bact_names_2.append(' '.join(bact['targetSpecies']['name'].split()[:2]))
                # else:
                #     print(AMP['id'])
                #     continue
            # bact_list.append(set(bact_names))
            # bact_list_2.append(set(bact_names_2))
            # for name in set(bact_names):
            #     if name not in list(bact_count.keys()):
            #         bact_count[name] = 1
            #     else:
            #         bact_count[name] += 1
            # for name in set(bact_names_2):
            #     if name not in list(bact_count_2.keys()):
            #         bact_count_2[name] = 1
            #     else:
            #         bact_count_2[name] += 1
        else:
            print(AMP['id'])
    print(huge_bact_name)
    print(measure_unit)

#### 查看所有细菌的单位

In [ ]:
from tqdm import tqdm
# for huge_bact_name in ['Pseudomonas aeruginosa PAO1']:
measure_unit = {}
for AMP in tqdm(data):
    bact_names = []
    bact_names_2 = []


    if AMP['targetActivities'] is not None:
        for bact in AMP['targetActivities']:

            if bact['targetSpecies'] is not None:
                # print(AMP['id'], 1)
                # if bact['targetSpecies']['name'] not in list(bact_count.keys()):
                #     bact_count[bact['targetSpecies']['name']] = 1
                # else:
                #     bact_count[bact['targetSpecies']['name']] += 1
                if bact['unit'] is not None:
                    if (bact['activityMeasureValue'], bact['unit']['name']) not in list(measure_unit.keys()):
                        measure_unit[(bact['activityMeasureValue'], bact['unit']['name'])] = 1
                    else:
                        measure_unit[(bact['activityMeasureValue'], bact['unit']['name'])] += 1
                    # if (bact['activityMeasureValue'], bact['unit']['name']) not in measure_unit:
                    #     measure_unit.append((bact['activityMeasureValue'], bact['unit']['name']))
                bact_names.append(bact['targetSpecies']['name'])
                bact_names_2.append(' '.join(bact['targetSpecies']['name'].split()[:2]))
            # else:
            #     print(AMP['id'])
            #     continue
        # bact_list.append(set(bact_names))
        # bact_list_2.append(set(bact_names_2))
        # for name in set(bact_names):
        #     if name not in list(bact_count.keys()):
        #         bact_count[name] = 1
        #     else:
        #         bact_count[name] += 1
        # for name in set(bact_names_2):
        #     if name not in list(bact_count_2.keys()):
        #         bact_count_2[name] = 1
        #     else:
        #         bact_count_2[name] += 1
    else:
        print(AMP['id'])
# print(huge_bact_name)
print(measure_unit)

#### notebook 以外步骤：从 concentration_unit_transfer_new.py 文件 来获得所有菌株对应的 MIC

#### 检查转换成 MIC 之后, SMILES 格式下每个菌株实际数据数量

In [ ]:
import pandas as pd

# 读取 .csv 文件
file_path = "/home/tianang/Projects/Synergy/DataPrepare/Data/DBAASP_id_SMILES_bact_MICs.csv"  # 替换为你的 .csv 文件路径
df = pd.read_csv(file_path)

# 打印列名
print("列名：", df.columns.tolist())

# 选择需要检查的列
column_name = "your_column_name"  # 替换为你的列名

for column_name in df.columns.tolist()[2:]:
    # 检查该列中大于 -1 的数目
    count = (df[column_name] > -1).sum()
    print(f"{column_name} 中大于 -1 的数目为：{count}")

#### 把 DBAASP_id_SMILES_bact_MICs.csv 中的 SMILES 换成 DBAASP 的 amino acid seq 供 APEX 使用

In [ ]:
import pandas as pd

# 获得原始 AMP list 中的 id 和 sequence 对
amino_acid_data = []
for AMP in data:
    # print(AMP['id'])
    if AMP['complexity']['name'] == 'Monomer':
        amino_acid_data.append({'id': AMP['id'], 'sequence': AMP['sequence']})
amino_acid_df = pd.DataFrame(amino_acid_data)
# print(amino_acid_data)

# 读取有 MIC 的 DataFrame
file_path = "/home/tianang/Projects/Synergy/DataPrepare/Data/DBAASP_id_same_as_AAseqs_SMILES_bact_MICs.csv"  # 替换为你的 .csv 文件路径
df = pd.read_csv(file_path)

# Amino acid 数据
# amino_acid_data = [
#     {'id': 1, 'sequence': 'VAL'},   # Valid canonical
#     {'id': 2, 'sequence': 'GLY'},   # Valid canonical
#     {'id': 3, 'sequence': 'ALA'},   # Valid canonical
#     {'id': 4, 'sequence': 'XVAL'}  # Noncanonical
# ]

# 将列表转换为 DataFrame
# amino_acid_df = pd.DataFrame(amino_acid_data)

# 定义 canonical 氨基酸的集合
canonical_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")  # 标准20种氨基酸的单字母代码

# 定义函数，检查序列是否只包含 canonical 氨基酸
def is_canonical(sequence):
    return all(char in canonical_amino_acids for char in sequence.strip())

# 合并两个 DataFrame，仅保留匹配的 DBAASP_id
df = df.merge(amino_acid_df, left_on='DBAASP_id', right_on='id')

# 过滤掉包含非canonical 氨基酸的行
df = df[df['sequence'].apply(is_canonical)]

# 更新列名并删除不需要的列
df.rename(columns={'sequence': 'AAseqs'}, inplace=True)
df.drop(columns=['SMILES', 'id'], inplace=True)
# df.drop(columns=['AAseqs', 'id'], inplace=True)

# 将 AAseqs 列移动到第二列
cols = df.columns.tolist()  # 获取当前列的列表
cols.insert(1, cols.pop(cols.index('AAseqs')))  # 将 'AAseqs' 从当前位置移动到索引 1
df = df[cols]  # 重新排列列顺序

# 重置索引，并删除旧索引列
df.reset_index(drop=True, inplace=True)

# 保存文件
output_path = "/home/tianang/Projects/Synergy/DataPrepare/Data/DBAASP_id_same_as_SMILES_AAseqs_bact_MICs.csv"
df.to_csv(output_path, index=False)
print(f'result saved to {output_path}')

# 打印结果
print(df)
print(len(df))

#### 上面的代码会去掉包含 X 等等非标准氨基酸的多肽，这里不管那么多直接替换就好

In [ ]:
import pandas as pd

# 获得原始 AMP list 中的 id 和 sequence 对
amino_acid_data = []
for AMP in data:
    # print(AMP['id'])
    if AMP['complexity']['name'] == 'Monomer':
        amino_acid_data.append({'id': AMP['id'], 'sequence': AMP['sequence'].strip()})
amino_acid_df = pd.DataFrame(amino_acid_data)
# print(amino_acid_data)

# 读取有 MIC 的 DataFrame
file_path = "/home/tianang/Projects/Synergy/DataPrepare/Data/DBAASP_id_SMILES_bact_MICs_512_limit.csv"  # 替换为你的 .csv 文件路径
df = pd.read_csv(file_path)

# Amino acid 数据
# amino_acid_data = [
#     {'id': 1, 'sequence': 'VAL'},   # Valid canonical
#     {'id': 2, 'sequence': 'GLY'},   # Valid canonical
#     {'id': 3, 'sequence': 'ALA'},   # Valid canonical
#     {'id': 4, 'sequence': 'XVAL'}  # Noncanonical
# ]

# 将列表转换为 DataFrame
# amino_acid_df = pd.DataFrame(amino_acid_data)

# 定义 canonical 氨基酸的集合
canonical_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")  # 标准20种氨基酸的单字母代码

# 定义函数，检查序列是否只包含 canonical 氨基酸
def is_canonical(sequence):
    return all(char in canonical_amino_acids for char in sequence.strip())

# 合并两个 DataFrame，仅保留匹配的 DBAASP_id
df = df.merge(amino_acid_df, left_on='DBAASP_id', right_on='id')

# 过滤掉包含非canonical 氨基酸的行
# df = df[df['sequence'].apply(is_canonical)]

# 更新列名并删除不需要的列
df.rename(columns={'sequence': 'AAseqs'}, inplace=True)
df.drop(columns=['SMILES', 'id'], inplace=True)
# df.drop(columns=['AAseqs', 'id'], inplace=True)

# 将 AAseqs 列移动到第二列
cols = df.columns.tolist()  # 获取当前列的列表
cols.insert(1, cols.pop(cols.index('AAseqs')))  # 将 'AAseqs' 从当前位置移动到索引 1
df = df[cols]  # 重新排列列顺序

# 重置索引，并删除旧索引列
df.reset_index(drop=True, inplace=True)

# 保存文件
output_path = "/home/tianang/Projects/Synergy/DataPrepare/Data/DBAASP_id_same_as_SMILES_AAseqs_bact_MICs_512_limit.csv"
df.to_csv(output_path, index=False)
print(f'result saved to {output_path}')

# 打印结果
print(df)
print(len(df))

#### 检查转换成 氨基酸序列 之后, AAseqs 格式下每个菌株实际数据数量

In [ ]:
import pandas as pd

# 读取 .csv 文件
file_path = "/home/tianang/Projects/Synergy/DataPrepare/Data/DBAASP_id_AAseqs_bact_MICs.csv"  # 替换为你的 .csv 文件路径
df = pd.read_csv(file_path)

# 打印列名
print("列名：", df.columns.tolist())

# 选择需要检查的列
column_name = "your_column_name"  # 替换为你的列名

for column_name in df.columns.tolist()[2:]:
    # 检查该列中大于 -1 的数目
    count = (df[column_name] > -1).sum()
    print(f"{column_name} 中大于 -1 的数目为：{count}")

#### 尝试比对两个分子
*注：python 文件版可以去看 visulize_mol_diff.py

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdFMCS
from rdkit.Chem import AllChem
import matplotlib.pyplot as plt
import re
from rdkit.Chem import rdDepictor

molA = Chem.MolFromSmiles("NCC(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)[OH]")
molB = Chem.MolFromSmiles("NCC(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CCC[C@H]1C(=O)N[C@@H](Cc1c[nH]c2ccccc12)C(=O)N[C@@H](Cc1c[nH]c2ccccc12)C(=O)N[C@@H](Cc1c[nH]c2ccccc12)C(=O)[OH]")

params = rdFMCS.MCSParameters()
params.BondCompare = rdFMCS.BondCompare.CompareOrder
params.RingMatchesRingOnly = True
params.MatchValences = True
print('matching two mols...')
mcs_result = rdFMCS.FindMCS([molA, molB], params)

# 首先为两个分子生成2D坐标
# AllChem.Compute2DCoords(molA)
# AllChem.Compute2DCoords(molB)

# 将B对齐到A上，利用最大公共子结构信息
# 1. 从 MCS 得到 SMARTS，并转换成分子对象

mcs_mol = Chem.MolFromSmarts(mcs_result.smartsString)
# 获取A、B中匹配公共子结构的原子索引
matchA = molA.GetSubstructMatch(mcs_mol)
matchB = molB.GetSubstructMatch(mcs_mol)
print(f'matchA: {matchA}')
print(f'matchB: {matchB}')
highlight_atoms_A = [i for i in range(molA.GetNumAtoms()) if i not in matchA]
highlight_atoms_B = [i for i in range(molB.GetNumAtoms()) if i not in matchB]

# 找出molA、molB中所有原子索引
allAtomsA = set(range(molA.GetNumAtoms()))
allAtomsB = set(range(molB.GetNumAtoms()))

molA_copy = Chem.Mol(molA)
molB_copy = Chem.Mol(molB)

diffAtomsA = set(range(molA_copy.GetNumAtoms())) - set(matchA)
diffAtomsB = set(range(molB_copy.GetNumAtoms())) - set(matchB)
# 公共部分原子
# 3. 使用2D对齐函数，让B的子结构对齐到A
# atomMap = [[b, a] for b, a in zip(matchB, matchA) if b < molB.GetNumAtoms() and a < molA.GetNumAtoms()]
# rdDepictor.GenerateDepictionMatching2DStructure(molA, molB, atomMap=atomMap)

# 用于在 SMILES 中标记不一样的
for idx in diffAtomsA:
    molA_copy.GetAtomWithIdx(idx).SetAtomMapNum(999)

for idx in diffAtomsB:
    molB_copy.GetAtomWithIdx(idx).SetAtomMapNum(999)

print(f'A_similarity: {len(matchA) / len(allAtomsA)*100}%')
print(f'B_similarity: {len(matchB) / len(allAtomsB)*100}%')

# 设置成 canonical=False 可以防止因为 SetAtomMapNum=999 导致原子在SMILES中的顺序被改变
smiA_marked = Chem.MolToSmiles(molA_copy, canonical=False)
smiB_marked = Chem.MolToSmiles(molB_copy, canonical=False)

pattern = r"\[[A-Za-z0-9@+\-\(\)=#]+:999\]"
matches = re.finditer(pattern, smiB_marked)
print([(match.start(), match.group()) for match in matches])

print(f'smiA_marked:\n {smiA_marked}\nsmiA_original:\n {Chem.MolToSmiles(molA, canonical=True)}')
print(f'smiB_marked:\n {smiB_marked}\nsmiB_original:\n {Chem.MolToSmiles(molB, canonical=True)}')

# 绘制并展示对齐后的结果
img = Draw.MolsToGridImage([molA, molB], molsPerRow=2, subImgSize=(1500,1500), highlightAtomLists=[highlight_atoms_A, highlight_atoms_B])
# 使用 matplotlib 展示图片
display(img)

#### 尝试获得不同的SMILES部分在哪个token中的代码在 bert-loves-chemistry 项目中，diff_tokenizer_try.py
#### *注：所有不同的 SMILES 通过 compare_all_mol_diff_parallel.py 获得，结果文件是 DBAASP_id_SMILES_compare.csv
#### *注：所有的比较 C@@H 和 C@H 的代码在 clean_smiles_compare.py 中完成，结果文件是 /Data/DBAASP_id_SMILES_compare_cleaned.csv
#### 以下代码读取看看那些完全一样的 SMILES 有多少

In [ ]:
import pandas as pd
df = pd.read_csv('./Data/DBAASP_id_SMILES_compare_cleaned_w_mean_MIC.csv')

filtered_df = df[(df["similarity_to_MCS_1"] == 1) & (df["similarity_to_MCS_2"] == 1)]
filtered_df = filtered_df.reset_index(drop=True)

for i in range(filtered_df.shape[0]):
    print(filtered_df.loc[i, 'smiles1'])
    print(filtered_df.loc[i, 'smiles2'])
    print('\n')

filtered_df

#### 构建完整比较smiles数据集, 去掉那些 mean MIC 相聚太近的数据点

In [ ]:
import pandas as pd

mean_MIC_df = pd.read_csv('./Data/DBAASP_id_SMILES_bact_mean_MICs.csv')

mean_MIC_data = mean_MIC_df.values

# 创建一个从 DBAASP_id 到 mean_MIC 映射的 dict
mean_MIC_dict = {DBAASP_id:mean_MIC for DBAASP_id, mean_MIC in zip(mean_MIC_data[:, 0], mean_MIC_data[:, -1])}

# 读取 SMILES compare记结果，并为其配上对应的 mean MIC
df = pd.read_csv('./Data/DBAASP_id_SMILES_compare_cleaned.csv')

origi_columns = list(df.columns)

fold_change = 0.5

smiles_comp_data = df.values
smiles_comp_data_w_mean_MIC = []
for line in smiles_comp_data:
    # print(line)
    # exit(0)
    mic_1 = mean_MIC_dict.get(line[0], None)
    mic_2 = mean_MIC_dict.get(line[1], None)
    if mic_1 and mic_2 is not None:
        # 还需要两个数值有一定的差距, 比如这里最小值必须是最大值 fold_change 以下
        if fold_change*max(mic_1, mic_2) > min(mic_1, mic_2):
            line = line.tolist()
            line.extend([mic_1, mic_2])
            smiles_comp_data_w_mean_MIC.append(line)
        # print(line)
        # break

print(f' length of filtered data: {len(smiles_comp_data_w_mean_MIC)}\n length of original data: {len(smiles_comp_data)}')
origi_columns.extend(['mean_MIC_1', 'mean_MIC_2'])
df = pd.DataFrame(smiles_comp_data_w_mean_MIC, columns=origi_columns)
# df
df.to_csv(f'./Data/DBAASP_id_SMILES_compare_cleaned_w_mean_MIC_{int(fold_change*100)}.csv', index=False)
# mean_MIC_dict
# mean_MIC_data

In [ ]:
df

#### 检查对比smiles这一步增加了多少种新的 peptide

In [ ]:
compared_DBAASP_id_set = set()
df = pd.read_csv('./Data/DBAASP_id_SMILES_compare_cleaned_w_mean_MIC.csv')
compared_data = df.values
for line in compared_data:
    line  = line.tolist()
    compared_DBAASP_id_set.update([line[0], line[1]])
# print(compared_DBAASP_id_set)
df_19_strains = pd.read_csv('./Data/DBAASP_id_SMILES_bact_MICs.csv')
df_w_mean = pd.read_csv('./Data/DBAASP_id_SMILES_bact_mean_MICs.csv')
all_ids = set(df_w_mean['DBAASP_id'].values)
_19_strain_DBAASP_id_set = set(df_19_strains['DBAASP_id'].values)
# print(_19_strain_DBAASP_id_set)
common_ids = compared_DBAASP_id_set & _19_strain_DBAASP_id_set
all_used_ids = compared_DBAASP_id_set | _19_strain_DBAASP_id_set

print(f' length of compared ids: {len(compared_DBAASP_id_set)}')
print(f' length of common ids: {len(common_ids)}')
print(f' length of 19 strains ids: {len(_19_strain_DBAASP_id_set)}')
print(f' length of all used pep ids: {len(all_used_ids)}')
print(f' length of all available pep ids: {len(all_ids)}')
# df

In [ ]:
import numpy as np

my_list = np.array([10, 20, 30, 40, 50, 20, 60, 30, 70, 20])
targets = np.array([20, 30, 60])  # 需要查找的元素

indices = np.where(np.isin(my_list, targets))[0]  # 找到匹配的下标
print(indices)  # 输出 [1 2 5 7 6]

#### 提取所有的 synergy 数据
*注：由外部文件完成 get_synergy.py

In [ ]:
for AMP in data:
    if len(AMP['synergies']) > 0:


In [ ]:
import torch
import torch.nn as nn

# 定义一个简单的转置卷积层
# 参数说明：
# - in_channels：输入通道数
# - out_channels：输出通道数
# - kernel_size：卷积核大小
# - stride：步幅
# - padding：填充（这里与普通卷积类似）
# - output_padding：输出时增加的额外尺寸（为了确保输出尺寸符合预期）
conv_transpose = nn.ConvTranspose2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    stride=2,
    padding=1,
    output_padding=1
)
# 1 1 1 1 1 1 1 1 1
# 打印转置卷积层的结构
print("转置卷积层结构:")
print(conv_transpose)

# 创建一个随机的输入张量
# 假设输入张量的尺寸为 (batch_size, channels, height, width) = (1, 1, 4, 4)
input_tensor = torch.randn(1, 1, 4, 4)
print("\n输入张量尺寸:", input_tensor.shape)

# 使用转置卷积层进行前向传播
output = conv_transpose(input_tensor)
print("输出张量尺寸:", output.shape)

#### 检查手动更新过的 MIC data 中现在有可用 genome 的数据点有多少了

In [ ]:
import json

with open('./Data/Evo_edition_2_MIC_data_handcrafted.json', 'r', encoding='utf-8') as f:
    data = json.load(f)  # 解析 JSON 文件

total_usable_genome = 0
for name, count in data.items():
    if 'ATCC' in name or '*' in name:
        # print(name)
        total_usable_genome += count

print(f' num of usable datapoints with genome: {total_usable_genome}')


#### 检查 download ATCC Genome 之后还有多少可用的数据点

In [ ]:
from pathlib import Path
import json

def get_stored_ATCC_IDs(folder_path):
    """
    检查哪些 ATCC ID 已经被下载了
    :param folder_path:
    :return:
    """
    stored_ATCC_IDs = []
    files = [f.name for f in folder_path.iterdir() if f.is_file()]
    for file_name in files:
        file_name = file_name.split('.')[0]
        file_name = file_name.split('ATCC')[-1]
        components = file_name.split('_')[1:]
        if len(components) == 2:
            stored_ATCC_IDs.append('-'.join(components))
        else:
            stored_ATCC_IDs.append(components[0])

    return stored_ATCC_IDs

folder_path = Path('./Data/Genome/ATCC')
stored_ATCC_IDs = get_stored_ATCC_IDs(folder_path)

with open('./Data/Evo_edition_2_MIC_data_handcrafted.json', 'r', encoding='utf-8') as f:
    strain_count_data = json.load(f)  # 解析 JSON 文件

total_usable_genome = 0
no_genome_strain_dict = {}

for name, count in strain_count_data.items():
    if '*' in name:
        new_name = name.split('*')[-1]
        if 'ATCC' not in new_name:
            total_usable_genome += count
        else:
            ATCC_id = new_name.split('ATCC')[-1].strip()
            if ATCC_id in stored_ATCC_IDs:
                total_usable_genome += count
            else:
                no_genome_strain_dict[name] = count
    if '*' not in name and 'ATCC' in name:
        ATCC_id = name.split('ATCC')[-1].strip()
        if 'BAA' in name:
            ATCC_id = ATCC_id.replace(" ", "-")
        if 'MY' in name:
            ATCC_id = ATCC_id.replace(" ", "")
        if 'MAY' in name:
            ATCC_id = ATCC_id.replace("MAY", "MYA")
        if 'D' in name:
            ATCC_id = ATCC_id.split("D")[0]
        if 'T' in name:
            ATCC_id = ATCC_id.split("T")[0]
        if 's' in name:
            ATCC_id = ATCC_id.split("s")[0]
        if " " in name:
            ATCC_id = ATCC_id.split(" ")[0]

        if ATCC_id in stored_ATCC_IDs:
            total_usable_genome += count
        else:
            no_genome_strain_dict[name] = count

print(f' unusable num of datapoints for each strain:\n{json.dumps(dict(sorted(no_genome_strain_dict.items(), key=lambda item: item[1], reverse=True)), indent=4)}')
print(f' num of usable datapoints with genome: {total_usable_genome}')

#### 获取所有strain的species name用于给树算距离

In [ ]:
from pathlib import Path

ATCC_fasta_folder_path = Path('./Data/Genome/ATCC')

file_names = [f.name for f in ATCC_fasta_folder_path.iterdir() if f.is_file()]

species_names = set()

for file_name in file_names:
    file_name = file_name.split('ATCC')[0]
    if 'subsp' in file_name.split('_'):
        file_name = file_name.split('subsp')[0]
    if 'pathovar' in file_name.split('_'):
        file_name = file_name.split('pathovar')[0]  # 带有 pathovar 和 var 的在 NCBI Taxonomy Browser 中都是识别不到的
    if 'var' in file_name.split('_'):
        file_name = file_name.split('var')[0]
    if 'sp' in file_name.split('_'):
        file_name = file_name.split('_sp')[0]
    species_name = file_name.replace('_', ' ').strip()
    species_names.add(species_name)

save_path = Path('./Data/Genome/visualization/species_names.txt')
with open(save_path, mode='w', encoding='utf-8') as f:
    for species_name in species_names:
        f.write(species_name + '\n')

#### 计算 species (树叶子节点) 之间的距离

In [ ]:
from Bio import Phylo
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# 读取 .phy 文件中的树
tree = Phylo.read("./Data/Genome/visualization/Phylogenetic_Tree_from_handcrafted.phy", "newick")  # 如果你的文件格式不是 Newick，请修改格式参数

species = np.array([clade.name for clade in tree.get_terminals()])

# 构建距离矩阵
n = len(species)
dist_matrix = np.zeros((n, n))
for i in tqdm(range(n), desc=' computing distance matrix'):
    for j in range(i, n):
        dist_matrix[i, j] = tree.distance(species[i], species[j])

dist_matrix = dist_matrix + dist_matrix.T

print(f' Drawing...')
# 可视化距离矩阵：绘制热力图
plt.figure(figsize=(80, 70))
sns.heatmap(dist_matrix, xticklabels=species, yticklabels=species, annot=True, cmap='viridis')
# plt.xticks(rotation=45, fontsize=5)
# plt.yticks(rotation=45, fontsize=5)
plt.title("species distance")
plt.savefig("./Data/Genome/visualization/phylo_distance.pdf")
plt.show()

#### 聚类不同的 species, 然后弄进去 iTOL 的模版画图
注：这里是利用 .phy 树文件来算距离进行聚类的，所以一定要处理好里面的名字，可以手动处理，比如去掉 '['，']' 防止不显示

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from Bio import Phylo
from tqdm import tqdm
import numpy as np
from pathlib import Path

current_dir = Path('/data2/tianang/projects/Synergy/DataPrepare')

# tree = Phylo.read("Data/Genome/visualization/Serratia_replaced_tree_Fungi_Enterobacterales.phy", "newick")  # TODO: 用于ablation study的分类tree
tree = Phylo.read(current_dir/"Data/Genome/visualization/All_species_gt_Taxonomy_Tree_cluster.phy", "newick")  # TODO: 用于文章主要内容的tree
# tree = Phylo.read(current_dir/"Data/Genome/visualization/All_species_gt_Taxonomy_Tree_draw.phy", "newick")  #TODO: All_species_gt_Taxonomy_Tree_cluster.phy 只有在分 11 类的时候比较合适
species = np.array([clade.name for clade in tree.get_terminals()])

# 构建距离矩阵
n = len(species)
dist_matrix = np.zeros((n, n))
for i in tqdm(range(n), desc=' computing distance matrix'):
    for j in range(i, n):
        dist_matrix[i, j] = tree.distance(species[i], species[j])

dist_matrix = dist_matrix + dist_matrix.T

# 假设你的预计算距离矩阵变量名为 distance_matrix，形状为 (n_samples, n_samples)
# 训练得到聚类
agg = AgglomerativeClustering(n_clusters=11, metric='precomputed', linkage='average')
labels = agg.fit_predict(dist_matrix)

# 打开 iTOL range 配置文件
with open(current_dir/'Data/Genome/visualization/range_template.txt', 'r') as read_f:
    template_content = read_f.read()

# range_setting_save_path = './Data/Genome/visualization/all_species_range_setting.txt'
range_setting_save_path = current_dir/'Data/Genome/visualization/range_setting.txt'

write_f = open(range_setting_save_path, 'w')

write_f.writelines(template_content)

all_grouped_species = []

labels_set = set(labels)
colors = [
    "FFFFCC",  # 浅黄色
    "FFEFC9",  # 奶油黄
    "FFD6D0",  # 淡珊瑚粉
    "F5C8E0",  # 粉紫色
    "eae3ff",  # 薰衣草紫
    "e3d0ff",  # 淡紫色
    "bacdff",  # 清透浅蓝（增强饱和度）
    "e4effa",  # 亮中浅蓝（比天空蓝更蓝）
    "F6CEDB",  # 樱花粉白
    "F9D8E3",  # 柔粉色
    "FAEDF2"   # 极浅樱粉白
] # 蓝，绿，橙，紫，红
range_line_format = "'{}','{}',#{}"  # strain_name, strain_name, color
for label in labels_set:
    indices = np.where(labels == label)[0]
    grouped_species = species[indices]
    all_grouped_species.append(grouped_species)
    print(f'\n Group {label}: {len(grouped_species)} species\n {grouped_species}')
    for _species in grouped_species:
        write_f.write(range_line_format.format(_species, _species, colors[label]) + '\n')

write_f.close()

print(f' Saved to {range_setting_save_path}')
# print(labels)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from Bio import Phylo
from tqdm import tqdm
import numpy as np
from pathlib import Path

current_dir = Path('/data2/tianang/projects/Synergy/DataPrepare')

# tree = Phylo.read("Data/Genome/visualization/Serratia_replaced_tree_Fungi_Enterobacterales.phy", "newick")  # TODO: 用于ablation study的分类tree
# tree = Phylo.read("Data/Genome/visualization/All_species_gt_Taxonomy_Tree_cluster.phy", "newick")  # TODO: 用于文章主要内容的tree
tree = Phylo.read(current_dir/"Data/Genome/visualization/All_species_gt_Taxonomy_Tree_draw.phy", "newick")  #TODO: All_species_gt_Taxonomy_Tree_cluster.phy 只有在分 11 类的时候比较合适
species = np.array([clade.name for clade in tree.get_terminals()])

# 构建距离矩阵
n = len(species)
dist_matrix = np.zeros((n, n))
for i in tqdm(range(n), desc=' computing distance matrix'):
    for j in range(i, n):
        dist_matrix[i, j] = tree.distance(species[i], species[j])

dist_matrix = dist_matrix + dist_matrix.T

# 假设你的预计算距离矩阵变量名为 distance_matrix，形状为 (n_samples, n_samples)
# 训练得到聚类
agg = AgglomerativeClustering(n_clusters=3, metric='precomputed', linkage='average')
labels = agg.fit_predict(dist_matrix)

# 打开 iTOL range 配置文件
with open(current_dir/'Data/Genome/visualization/range_template.txt', 'r') as read_f:
    template_content = read_f.read()

# range_setting_save_path = './Data/Genome/visualization/all_species_range_setting.txt'
range_setting_save_path = current_dir/'Data/Genome/visualization/range_setting.txt'

write_f = open(range_setting_save_path, 'w')

write_f.writelines(template_content)

all_grouped_species = []

labels_set = set(labels)
# colors = [
#     "F9F8D6",  # 浅黄色
#     "F7EADB",  # 淡紫色
#     "F5DCE0"   # 极浅樱粉白
# ] # 蓝，绿，橙，紫，红
colors = ["FFFFCC", "F9D8E3", "e3d0ff"]
colors = ["e3d0ff", "FFFFCC", "F9D8E3"]
range_line_format = "'{}','{}',#{}"  # strain_name, strain_name, color
for label in labels_set:
    indices = np.where(labels == label)[0]
    grouped_species = species[indices]
    all_grouped_species.append(grouped_species)
    print(f'\n Group {label}: {len(grouped_species)} species\n {grouped_species}')
    for _species in grouped_species:
        write_f.write(range_line_format.format(_species, _species, colors[label]) + '\n')

write_f.close()

print(f' Saved to {range_setting_save_path}')
# print(labels)

#### 估算每一个聚类里面有多少数据
直接统计每一个species大概有多少

In [ ]:
from pathlib import Path
import json

ATCC_fasta_folder_path = Path('./Data/Genome/ATCC')

file_names = [f.name for f in ATCC_fasta_folder_path.iterdir() if f.is_file()]

species_names = set()

for file_name in file_names:
    file_name = file_name.split('ATCC')[0]
    if 'subsp' in file_name:
        file_name = file_name.split('subsp')[0]
    species_name = file_name.replace('_', ' ').strip()

    # 'sp' 有时候处理会出错，需要特殊处理一下
    if species_name.split()[-1] == 'sp':
        species_name += '.'
    species_names.add(species_name)

with open('./Data/Evo_edition_2_MIC_data_handcrafted.json', 'r', encoding='utf-8') as file:
    data_dict = json.load(file)

species_count_dict = {}
for name, count in data_dict.items():
    if '*' in name or 'ATCC' in name:
        for species_name in species_names:
            if species_name in name:

                # 专门处理 'sp.' 的代码，'sp.' 只适用于匹配而已，用完之后就去掉
                if '.' in species_name:
                    species_name= species_name.split('sp.')[0].strip()
                if species_name not in species_count_dict.keys():
                    species_count_dict[species_name] = count
                else:
                    species_count_dict[species_name] += count

print(f' species count:\n{json.dumps(species_count_dict, indent=4)}')

for i, grouped_species in enumerate(all_grouped_species):
    group_data_count = 0
    print(grouped_species)
    for species_name in grouped_species:
        if species_count_dict.get(species_name) is not None:
            group_data_count += species_count_dict.get(species_name)
        # else:
        #     print(f' {species_name} not found in counting dict')
    print(f'\n group {i} data points: {group_data_count}\n')


#### 检查有没有重复的 SMILES

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem

df = pd.read_csv('./Data/DBAASP_id_SMILES_merged.csv')

DBAASPid_SMILES = np.array(df[['DBAASP_id', 'SMILES']].values.tolist())

for i, (DBAASP_id, smiles) in tqdm(enumerate(DBAASPid_SMILES), desc=' comparing ', total=len(DBAASPid_SMILES)):
    for j in range(i+1, len(DBAASPid_SMILES)):
        if len(smiles) == len(DBAASPid_SMILES[j][1]):
            mol_A = Chem.MolFromSmiles(smiles)
            mol_B = Chem.MolFromSmiles(DBAASPid_SMILES[j][1])
            if mol_B is None:
                print(f' DBAASP id: {DBAASPid_SMILES[j][0]}, SMILES: {DBAASPid_SMILES[j][1]}')
            canonical_smiles_A = Chem.MolToSmiles(mol_A)
            canonical_smiles_B = Chem.MolToSmiles(mol_B)
            if canonical_smiles_A == canonical_smiles_B:
                print(f' same smiles DBAASP ID: {DBAASP_id} | {DBAASPid_SMILES[j][0]}')

#### 检查 DBAASP 里面的 amino acid 序列有没有和 inhouse data 重合的

In [ ]:
from tqdm import tqdm
import pandas as pd

in_house_data = pd.read_csv("./Data/APEX 1.1 Data.csv").values

same_inhouse = []

for AMP in tqdm(data, desc=' comparing ', total=len(data)):
    if AMP['complexity']['name'] == 'Monomer':
        for inhouse_id, inhouse_amp in enumerate(in_house_data[:, 0]):
            if inhouse_amp == AMP['sequence']:
                same_inhouse.append(inhouse_id)
                print(f'\n In house ID {inhouse_id} and DBAASP ID {AMP["id"]} are the same')

print(len(same_inhouse))
print(len(set(same_inhouse)))

#### 获取所有新增加的strain的species name用于给树算距离

In [ ]:
from pathlib import Path

ATCC_fasta_folder_path = Path('./Data/Text_Description/wo_ATCC/Text')

file_names = [f.name for f in ATCC_fasta_folder_path.iterdir() if f.is_file()]

species_names = set()

for file_name in file_names:
    # file_name = file_name.split('ATCC')[0]
    # if 'subsp' in file_name.split('_'):
    #     file_name = file_name.split('subsp')[0]
    # if 'pathovar' in file_name.split('_'):
    #     file_name = file_name.split('pathovar')[0]  # 带有 pathovar 和 var 的在 NCBI Taxonomy Browser 中都是识别不到的
    # if 'var' in file_name.split('_'):
    #     file_name = file_name.split('var')[0]
    # if 'sp' in file_name.split('_'):
    #     file_name = file_name.split('_sp')[0]
    species_name = ' '.join(file_name.split('.txt')[0].split('～')[:2])
    if species_name.split(' ')[-1] in ['sp.', 'spp.', 'group']:
        continue
    species_names.add(species_name)

save_path = Path('./Data/Genome/visualization/text_only_species_names.txt')
with open(save_path, mode='w', encoding='utf-8') as f:
    for species_name in species_names:
        f.write(species_name + '\n')

#### ATCC带genome的和获取所有新增加的只有text的strain的species name用于给树算距离

In [ ]:
from pathlib import Path

species_names = set()

ATCC_fasta_folder_path = Path('./Data/Genome/ATCC')

file_names = [f.name for f in ATCC_fasta_folder_path.iterdir() if f.is_file()]

for file_name in file_names:
    file_name = file_name.split('ATCC')[0]
    if 'subsp' in file_name.split('_'):
        file_name = file_name.split('subsp')[0]
    if 'pathovar' in file_name.split('_'):
        file_name = file_name.split('pathovar')[0]  # 带有 pathovar 和 var 的在 NCBI Taxonomy Browser 中都是识别不到的
    if 'var' in file_name.split('_'):
        file_name = file_name.split('var')[0]
    if 'sp' in file_name.split('_'):
        file_name = file_name.split('_sp')[0]
    species_name = file_name.replace('_', ' ').strip()
    species_names.add(species_name)

ATCC_fasta_folder_path = Path('./Data/Text_Description/wo_ATCC/Text')

file_names = [f.name for f in ATCC_fasta_folder_path.iterdir() if f.is_file()]

for file_name in file_names:
    # file_name = file_name.split('ATCC')[0]
    # if 'subsp' in file_name.split('_'):
    #     file_name = file_name.split('subsp')[0]
    # if 'pathovar' in file_name.split('_'):
    #     file_name = file_name.split('pathovar')[0]  # 带有 pathovar 和 var 的在 NCBI Taxonomy Browser 中都是识别不到的
    # if 'var' in file_name.split('_'):
    #     file_name = file_name.split('var')[0]
    # if 'sp' in file_name.split('_'):
    #     file_name = file_name.split('_sp')[0]
    species_name = ' '.join(file_name.split('.txt')[0].split('～')[:2])
    if species_name.split(' ')[-1] in ['sp.', 'spp.', 'group']:
        continue
    species_names.add(species_name)

save_path = Path('./Data/Genome/visualization/all_species_names.txt')
with open(save_path, mode='w', encoding='utf-8') as f:
    for species_name in species_names:
        f.write(species_name + '\n')

#### 处理 Nature 的数据成 Evo 的格式

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

raw_folder_path = Path('./Data/small_molecule/raw')
raw_file_path = raw_folder_path / 'nature_1_positive~Staphylococcus_aureus_RN4220.csv'

df = pd.read_csv(raw_file_path)

raw_data = df.values

processed_data = []
for i, line in tqdm(enumerate(raw_data), total=len(raw_data)):
    id = f'{raw_file_path.name[0:2]}_{i}'
    strain = 'Staphylococcus aureus RN4220'
    smiles = line[0]
    label = line[1]
    processed_data.append([id, strain, smiles, label])

columns = ['DBAASP_id','strain_name','SMILES','MIC']

processed_folder_path = Path('./Data/small_molecule/processed')
processed_file_path = processed_folder_path / 'nature_1_positive~Staphylococcus_aureus_RN4220.csv'

pd.DataFrame(processed_data, columns = columns).to_csv(processed_file_path, index=False, mode='w')

#### 处理 Nature Chem Bio 的数据成 Evo 的格式

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import numpy as np

raw_folder_path = Path('./Data/small_molecule/raw')
raw_file_path = raw_folder_path / 'chem_bio_relative_growth~Acinetobacter_baumannii_ATCC_17978.csv'

df = pd.read_csv(raw_file_path)

raw_data = df.values

relative_growth = raw_data[:, -1]

mean_val = np.mean(relative_growth)
std_val = np.std(relative_growth, ddof=1)

labels = (relative_growth < (mean_val - std_val)).astype(int)

processed_data = []
for i, (smiles, label) in tqdm(enumerate(zip(raw_data[:, 0], labels)), total=len(raw_data)):
    id = f'{raw_file_path.name[0:2]}_{i}'
    strain = '17978'
    # smiles = line[0]
    processed_data.append([id, strain, smiles, label])

columns = ['DBAASP_id','strain_name','SMILES','MIC']

processed_folder_path = Path('./Data/small_molecule/processed')
processed_file_path = processed_folder_path / 'chem_bio_relative_growth~Acinetobacter_baumannii_ATCC_17978.csv'

pd.DataFrame(processed_data, columns = columns).to_csv(processed_file_path, index=False, mode='w')

#### 处理 Cell 的数据成 Evo 的格式

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

raw_folder_path = Path('./Data/small_molecule/raw')
raw_file_path = raw_folder_path / 'cell~Escherichia_coli_BW25113~#004.csv'

df = pd.read_csv(raw_file_path)

raw_data = df.values

processed_data = []
for i, line in tqdm(enumerate(raw_data), total=len(raw_data)):
    id = f'{raw_file_path.name[0:2]}_{i}'
    strain = '#004'
    smiles = line[1]
    label = 1 if line[-1] == 'Active' else 0
    processed_data.append([id, strain, smiles, label])

columns = ['DBAASP_id','strain_name','SMILES','MIC']

processed_folder_path = Path('./Data/small_molecule/processed')
processed_file_path = processed_folder_path / 'cell~Escherichia_coli_BW25113~#004.csv'

pd.DataFrame(processed_data, columns = columns).to_csv(processed_file_path, index=False, mode='w')

#### 合并三个 数据集

In [ ]:
import pandas as pd
from pathlib import Path
processed_folder_path = Path('./Data/small_molecule/processed')
file_names = [f.name for f in processed_folder_path.iterdir()]
df_list = [pd.read_csv(processed_folder_path / f) for f in file_names]
all_df = pd.concat(df_list, ignore_index=True)
all_df.to_csv(processed_folder_path / 'small_molecule_Evo_binary_data.csv', index=False, mode='w')

#### 画 MIC 分布

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/DBAASP_inhouse_AMP_SMILES_MIC_Evo.csv')

# 2. 提取指定列的数据（假设列名为 'column_name'）
#    并去除可能的缺失值
values = df['MIC'].dropna().values
data = df.values
count = 0
for line in data:
    if line[-1] >= 128:
        count += 1


values = values[values < 20000]
print(count)
print(len(data)-count)
print(f' median:{np.median(values)}')
print(f' mean:{np.mean(values)}')
# values = values[values < 1000]

# 3. 绘制分布图（例如直方图）
plt.figure(figsize=(8, 5))
plt.yscale('log')
plt.hist(values, bins=300, edgecolor='black')  # bins 可调整柱子数量
plt.title(' MIC distribution')
plt.xlabel('MIC')
plt.ylabel('count')
plt.grid(True, linestyle='--', alpha=0.5)      # 可选：加网格方便阅读
plt.show()

#### 检查一共有多少的 AMP strain pair

In [ ]:
import pandas as pd

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/DBAASP_inhouse_AMP_SMILES_MIC_Evo.csv')

print(len(df))

#### 检查一共有多少的 small molecule strain pair

In [ ]:
import pandas as pd

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/small_molecule/processed/small_molecule_Evo_binary_data_SELFIES.csv')

print(len(df))

#### 检查 synergy 数据中有多少是 AMP-AMP 的

In [ ]:
import pandas as pd

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/synergistic_pairs_Evo.csv')

data = df.values[:, 1]

def is_int(s):
    try:
        int(s)
        return True
    except ValueError:
        return False

AMP_count = 0
for id in data:
    if is_int(id):
        AMP_count += 1

print(AMP_count)
print(len(data))

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
import selfies as sf

model_name = "ibm-research/materials.selfies-ted"
tokenizer = AutoTokenizer.from_pretrained(model_name)

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/DBAASP_id_SELFIES_bact_MICs.csv')

SELFIES = df['SMILES'].values

SELFIES = [sf.encoder('O=C(N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@@H]1C(=O)N[C@H](C(=O)N[C@@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)N[C@H](C(=O)NCC1)[C@H](O)C)CCN)CCN)CC(C)C)CC(C)C)CCN)CCN)[C@H](O)C)CCN)CCCC(C)CC')]

lens = []
for SELFIES_str in tqdm(SELFIES):
    # _len = len(SELFIES_str.replace('][', '] [').split(' '))
    _len = len(tokenizer(SELFIES_str.replace('][', '] ['))['input_ids'])
    lens.append(_len)

lens = np.array(lens)
print(f' mean: {np.mean(lens)}')
print(f' median: {np.median(lens)}')

In [ ]:
import pandas as pd

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/DBAASP_inhouse_AMP_SMILES_MIC_Evo.csv')

#### 统计 inhouse strain 和 unique molecule数量

In [ ]:
import pandas as pd

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/DBAASP_inhouse_AMP_SMILES_MIC_Evo.csv')
print(df.columns)
print(len(df))
print(len(set(df['strain_name'].values)))
print(len(set(df['SMILES'].values)))

#### 打印所有的 residue 和对应的 SMILES

In [ ]:
import pandas as pd

df = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/all_aa_smiles_new_handcrafted.csv')
aa_data = df.values
for line in aa_data:
    print(': '.join(line))

#### 打印所有 ATCC genome 的名字

In [ ]:
from pathlib import Path

folder = Path('/data2/tianang/projects/Synergy/DataPrepare/Data/Genome/ATCC')
aa_list = []
for file in folder.iterdir():
    aa_list.append(str(file.name).split('.fasta')[0].replace('_', ' '))

print(', '.join(aa_list))

In [ ]:
from pathlib import Path

folder = Path('/data2/tianang/projects/Synergy/DataPrepare/Data/Genome/ATCC')
aa_list = []
for file in folder.iterdir():
    print(str(file.name).split('.fasta')[0].replace('_', ' '))

# print(', '.join(aa_list))

In [ ]:
from rdkit import Chem
mol = Chem.MolFromSmiles('OC(=O)[C@@H](CCC(N)=O)N')
Chem.AssignStereochemistry(mol, force=True, cleanIt=True)
cip = mol.GetAtomWithIdx(3).GetProp('_CIPCode')   # → 'S'
cip

### 把小分子的处理成 SELFIES 的 .txt 文件给到mdlm那边做 screening

In [ ]:
import pandas as pd

SMILES_SM_raw = pd.read_csv('/data2/tianang/projects/Synergy/DataPrepare/Data/small_molecule/processed/small_molecule_Evo_binary_data.csv')['SMILES'].values
print(SMILES_SM_raw[0])
